In [ ]:
# =====================================================================================
#  Bajaj 3W benchmarking dashboard  -  AUTO-READING VERSION
#  Drop any benchmarking workbook(s) in the folder, run  run()  and the dashboard rebuilds.
#  Nothing is tied to a cell address any more: layout, header row, brands, vehicles,
#  parameters, peer sets and segments are all detected from the sheets themselves.
#  New rows / new columns / new vehicles / new sheets / new workbooks are picked up automatically.
# =====================================================================================
# !pip install openpyxl     (or  %pip install openpyxl  inside Jupyter)

import re, os, json, html, math, difflib, datetime, warnings
from collections import OrderedDict, defaultdict

# ------------------------------------------------------------------ CONFIG (edit freely)
# Anything matching this (in brand OR vehicle name) is treated as a Bajaj model.
BAJAJ_NAME_HINTS = r"\bbajaj\b|\bgogo\b|\bmaxima\b|\briki\b|\bbal\b|\bre\s?(compact|\d{3})"
# Vehicles whose name matches are ignored entirely (e.g. r"^BAL 4W DV").
EXCLUDE_VEHICLES = r""
# Name patterns that force a fuel / use when the sheets do not make it obvious.
EV_NAME_HINTS = r"\bev\b|electric|electrik|\bbev\b|\be[- ]?(rick|cart|city|xtra|loader|auto)\b|\bgogo\b|\briki\b|\btreo\b|\bzor\b"
# Manual overrides: list of (regex on "file | sheet | vehicle name", "Cargo" or "Passenger")
USE_OVERRIDES = []          # e.g. [(r"xwide", "Passenger")]
# Manual peer links for competitor-only sheets: {regex on "file | sheet": regex on Bajaj model name}
LINKS = {}                  # e.g. {r"p5009": r"P5009"}
# Direction (best = hi/lo) for NEW parameters you want highlighted: [(regex on label, "hi"|"lo")]
DYNAMIC_DIRECTION = []      # e.g. [(r"efficiency", "hi"), (r"turning", "lo")]
# Extra brands the detector should know (alias lower-case -> display name). New brands in a
# 'Brand / Make / OEM' row of a sheet are learned automatically.
EXTRA_BRANDS = {}
# Force a header row for a sheet: {("workbook.xlsx", "Sheet name"): 5}   (1-based Excel row)
HEADER_ROW_OVERRIDES = {}
MIN_FIELDS = 4              # a competitor needs at least this many filled parameters
MAX_ROWS, MAX_COLS = 600, 150
FILE_EXCLUDE = r"^~\$|dashboard|bajaj_benchmarking"

# ------------------------------------------------------------------ cleaning & parsing helpers
_NA = ('-', '--', '---', 'na', 'n/a', 'nan', 'tbc', 'to be confirmed', 'to be checked', 'not provided', 'none', 'nil', '?')

def S(v):
    if v is None: return None
    if isinstance(v, bool): return 'Yes' if v else 'No'
    if isinstance(v, float):
        if math.isnan(v): return None
        v = int(v) if v.is_integer() else round(v, 4)
    if isinstance(v, (datetime.datetime, datetime.date, datetime.time)): v = str(v)
    s = re.sub(r'\s+', ' ', str(v).replace('\u200b', '').replace('\xa0', ' ')).strip()
    if not s or s.startswith('#') or s.lower() in _NA: return None
    return s

def f(s):
    s = S(s)
    if s is None: return None
    s = re.sub(r'(?<=\d),(?=\d{3}(?!\d))', '', s)          # 1,200 -> 1200
    s = re.sub(r'(?<=\d),(?=\d)', '.', s)                   # 9,1  -> 9.1
    m = re.search(r'\d+(?:\.\d+)?', s)
    return float(m.group()) if m else None

def pw(s, label=''):                                        # -> kW  (handles kW / PS / HP / W)
    n = f(s)
    if n is None: return None
    low, lab = s.lower(), (label or '').lower()
    if 'kw' in low: return n
    if re.search(r'\bps\b', low) or re.search(r'\(.*\bps\b.*\)|\bps\b', lab): return round(n * 0.7355, 2)
    if re.search(r'\b(b?hp)\b', low) or re.search(r'\b(b?hp)\b', lab): return round(n * 0.7457, 2)
    if re.search(r'\d\s*w\b', low): return round(n / 1000, 3)
    return n

def nm(s, label=''):                                        # -> Nm
    n = f(s)
    if n is None: return None
    low = s.lower() + ' ' + (label or '').lower()
    if re.search(r'kg\s*[.\-]?\s*m|kgm', low): return round(n * 9.80665, 1)
    if 'wheel' in low or n > 300: return None               # wheel-level torque is not comparable
    return n

def gr(s, label=''):                                        # -> gradient %
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', s)
    if m: return float(m.group(1))
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:deg|°)', s, re.I)
    if m: return round(math.tan(math.radians(float(m.group(1)))) * 100, 1)
    n = f(s)
    if n is None: return None
    return round(n * 100, 1) if n <= 1 else n

def tm(s, label=''):                                        # -> hours
    m = re.search(r'(\d+)\s*:\s*(\d+)', s)
    if m: return round(int(m.group(1)) + int(m.group(2)) / 60, 2)
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:hrs?|hours?)(?:\s*(\d{1,2})(?!\d)\s*(?:min\w*)?)?', s, re.I)
    if m:
        mins = int(m.group(2)) if m.group(2) and int(m.group(2)) < 60 else 0
        return round(float(m.group(1)) + mins / 60, 2)
    m = re.search(r'(\d+)\s*min', s, re.I)
    if m: return round(int(m.group(1)) / 60, 2)
    return float(s) if re.fullmatch(r'\d+(?:\.\d+)?', s) else None

def wk(s, label=''):                                        # warranty -> km
    s = re.sub(r'(?<=\d) (?=\d{3})', '', s).replace(',', '')
    m = re.search(r'(\d+(?:\.\d+)?)\s*(lakh|lac|l|k)?\s*kms?', s, re.I)
    if not m: return None
    n = float(m.group(1)); u = (m.group(2) or '').lower()
    return n * (100000 if u in ('lakh', 'lac', 'l') else 1000 if u == 'k' else 1)

def price(s, label=''):                                     # -> Rs lakh
    n = f(s)
    return round(n / 1e5, 2) if n and n > 1000 else n

def cc(s, label=''):
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:cc|cm)', s, re.I)
    if m: return float(m.group(1))
    return f(s) if re.fullmatch(r'[\d.,]+', s) else None

def tank(s, label=''):
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:l|ltr|ltrs|litres?|liters?)\b', s, re.I)
    return float(m.group(1)) if m else f(s)

PARSE = {'f': lambda s, l='': f(s), 'price': price, 'pw': pw, 'nm': nm, 'gr': gr, 'tm': tm, 'wk': wk, 'cc': cc, 'tank': tank}

# ------------------------------------------------------------------ canonical parameter dictionary
# (key, display label, category, regex on lower-case row label, parser, best-direction, text fallback key)
# First match wins, so specific patterns sit above general ones. Anything NOT matched is still
# shown automatically as a new parameter - this table only adds clean units, ordering and highlighting.
C_PR, C_EN, C_CP, C_EA, C_DM, C_CH, C_FT, C_DF, C_WR = ('Price & range', 'Engine & motor', 'Capacity & performance',
    'Earning potential (35 km/day, Rs 3/kg freight)', 'Dimensions & space', 'Chassis', 'Features', 'Differentiators & notes', 'Warranty')
CANON_ROWS = [
 ('earn', 'Earning per day (Rs)', C_EA, r'earning', 'f', 'hi'),
 ('opcost', 'Daily operating cost (Rs)', C_EA, r'operating cost|running cost|opex', 'f', 'lo'),
 ('emi', 'EMI per day (Rs)', C_EA, r'\bemi\b', 'f', 'lo'),
 ('saving', 'Saving per day (Rs)', C_EA, r'saving', 'f', 'hi'),
 ('invest', 'Initial investment (Rs)', C_EA, r'initial invest|investment', 'f', 'lo'),
 ('payback', 'Payback (months)', C_EA, r'pay ?back', 'f', 'lo'),
 ('wty', 'Warranty terms', C_WR, r'warrant', 'wty', None),
 ('remarks', 'Remarks', C_DF, r'^(remarks?|comments?|notes?)\b', 't', None),
 ('diff', 'Differentiators', C_DF, r'differentiat|\busp\b|key (features?|highlights?)', 't', None),
 ('country', 'Country supplied', C_DF, r'country|origin', 't', None),
 ('dealers', 'Dealerships', C_DF, r'dealer', 't', None),
 # --- performance / capacity (before features so 'gradeability' etc. are not swallowed)
 ('grade', 'Gradeability (%)', C_CP, r'grade', 'gr', 'hi'),
 ('pay', 'Payload (kg)', C_CP, r'payload|load(ing)? capacity|carrying capacity|max(imum)? load', 'f', 'hi'),
 ('tray', 'Load tray (mm)', C_DM, r'load (body|bed|deck|tray|platform)|cargo (body|box|deck|bed)|tray|deck size', 't', None),
 ('cft', 'Cargo volume (cu ft)', C_CP, r'cargo volume|load volume|cu\.? ?ft|\bvolume\b', 'f', 'hi'),
 ('gvw', 'GVW (kg)', C_CP, r'\bgvw\b|gross vehicle', 'f', None),
 ('kerb', 'Kerb weight (kg)', C_CP, r'kerb|curb|unladen', 'f', 'lo'),
 ('spd', 'Top speed (km/h)', C_CP, r'top speed|max(imum)? speed|vmax|^speed\b', 'f', 'hi'),
 ('accel', 'Acceleration', C_CP, r'accelerat', 't', None),
 ('gc', 'Ground clearance (mm)', C_CP, r'ground clearance|\bgc\b', 'f', 'hi'),
 ('wb', 'Wheelbase (mm)', C_DM, r'wheel ?base', 'f', None),
 ('tcr', 'Turning radius', C_DM, r'turning', 't', None),
 ('wading', 'Water wading', C_CP, r'wading', 't', None),
 ('dim', 'Vehicle LxWxH (mm)', C_DM, r'dimension|overall size|vehicle size|l ?[x×*] ?w ?[x×*] ?h', 't', None),
 ('len', 'Length', '_h', r'overall length|^length', 'f', None),
 ('wid', 'Width', '_h', r'overall width|^width', 'f', None),
 ('hei', 'Height', '_h', r'overall height|^height', 'f', None),
 # --- features
 ('remimm', 'Remote immobilisation', C_FT, r'remote immob', 't', None),
 ('geofence', 'Geofencing / immobilisation', C_FT, r'geo ?fenc|immobili', 't', None),
 ('overspeed', 'Overspeed alert', C_FT, r'over ?speed', 't', None),
 ('limiter', 'Speed limiter', C_FT, r'speed limit', 't', None),
 ('shareloc', 'Share live location', C_FT, r'(share|live).*location', 't', None),
 ('chgalerts', 'Charging alerts', C_FT, r'charg(ing|e) alert', 't', None),
 ('tele', 'Telematics', C_FT, r'telematic|\bgps\b|\biot\b|connected', 't', None),
 ('revsens', 'Reverse parking sensor', C_FT, r'reverse.*sens|rear.*sens|parking sens', 't', None),
 ('parkassist', 'Park assist', C_FT, r'park(ing)? assist', 't', None),
 ('rollover', 'Roll-over detection', C_FT, r'roll ?over', 't', None),
 ('hazard', 'Auto hazard', C_FT, r'hazard', 't', None),
 ('bhot', 'Battery overheat alarm', C_FT, r'overheat', 't', None),
 ('wheelcover', 'Wheel cover', C_FT, r'wheel ?cover', 't', None),
 ('signature', 'Signature lamp', C_FT, r'signature|\bdrl\b', 't', None),
 ('cabin', 'Cabin light', C_FT, r'cabin (light|lamp)|interior light', 't', None),
 ('lamp', 'Lamps', C_FT, r'head ?lamp|^lamps?\b|lamps?$|lighting', 't', None),
 ('icl', 'Instrument cluster', C_FT, r'cluster|instrument', 't', None),
 ('speedo', 'Speedometer', C_FT, r'speedo', 't', None),
 ('hill', 'Hill hold', C_FT, r'hill', 't', None),
 ('usb', 'USB charger', C_FT, r'usb|mobile charg', 't', None),
 ('docking', 'Mobile docking', C_FT, r'dock', 't', None),
 ('fascia', 'Front fascia', C_FT, r'fascia', 't', None),
 ('hvac', 'HVAC / AC', C_CH, r'hvac|air ?cond|\bac\b', 't', None),
 ('music', 'Music system provision', C_FT, r'music|audio|infotainment', 't', None),
 ('idss', 'IDSS', C_FT, r'idss', 't', None),
 ('faremeter', 'Digital faremeter', C_FT, r'fare ?meter', 't', None),
 ('revlever', 'Reverse gear lever', C_FT, r'reverse (gear|lever)', 't', None),
 ('handstart', 'Hand start lever', C_FT, r'hand ?start', 't', None),
 ('parking', 'Parking brake', C_CH, r'parking brake|park brake', 't', None),
 ('modes', 'Drive modes', C_FT, r'drive modes?|riding modes?|\bmodes?\b', 't', None),
 ('regen', 'Regenerative braking', C_EN, r'regen', 't', None),
 ('grab', 'Grab handle', C_FT, r'grab', 't', None),
 ('hood', 'Hood frame', C_FT, r'hood', 't', None),
 ('step', 'Step-in height (mm)', C_FT, r'step[- ]in', 't', None),
 ('corr', 'Corrosion prevention', C_FT, r'corros', 't', None),
 ('weather', 'Weather protection', C_FT, r'weather', 't', None),
 ('utility', 'Utility space', C_FT, r'utility space', 't', None),
 # --- chassis & interior
 ('tubeless', 'Tubeless tyre', C_CH, r'tubeless', 't', None),
 ('abs', 'ABS', C_CH, r'\babs\b', 't', None),
 ('adjseat', 'Adjustable driver seat', C_CH, r'adjustable.*seat|seat adjust', 't', None),
 ('pseat', 'Passenger seat', C_DM, r'passenger seat', 't', None),
 ('backrest', 'Backrest to B-pillar (mm)', C_DM, r'backrest', 't', None),
 ('luggage', 'Rear luggage space', C_DM, r'luggage|boot space', 't', None),
 ('seat', 'Seating', C_DM, r'seating|no\.? of seats|passenger capacity|^seats?( capacity)?( \(.*\))?$', 't', None),
 ('spare', 'Spare wheel location', C_CH, r'spare', 't', None),
 ('jack', 'Jack location', C_CH, r'jack', 't', None),
 ('limp', 'Limp-home tank', C_CH, r'limp', 't', None),
 ('storage', 'Under-seat storage', C_CH, r'storage', 't', None),
 ('mudf', 'Front mudguard', C_CH, r'mud ?guard.*front|front.*mud', 't', None),
 ('mudr', 'Rear mudguard', C_CH, r'mud ?guard.*rear|rear.*mud', 't', None),
 ('door', 'Side door', C_CH, r'\bdoors?\b', 't', None),
 ('steer', 'Steering', C_CH, r'steering', 't', None),
 ('shaft', 'Drive shaft', C_CH, r'shaft', 't', None),
 ('fs', 'Front suspension', C_CH, r'front.*suspension|suspension.*front', 't', None),
 ('rs', 'Rear suspension', C_CH, r'rear.*suspension|suspension.*rear', 't', None),
 ('brk', 'Brakes', C_CH, r'brake', 't', None),
 ('tyre', 'Tyre', C_CH, r'tyre|tire', 't', None),
 ('body', 'Body / chassis', C_CH, r'body (structure|type)|chassis|construction', 't', None),
 # --- price, range, battery, charging
 ('price', 'Ex-showroom (Rs lakh)', C_PR, r'price|ex[- ]?showroom|\bmrp\b', 'price', None),
 ('reserve', 'Reserve range', C_PR, r'reserve', 't', None),
 ('rng', 'Range (km)', C_PR, r'\brange\b|\bidc\b', 'f', 'hi'),
 ('batt', 'Battery type', C_PR, r'battery (type|chemistry)|chemistry|cell type', 't', None),
 ('volt', 'Battery voltage', C_PR, r'voltage|\bvolts?\b', 't', None),
 ('bat', 'Battery (kWh)', C_PR, r'^batter(y|ies)( pack)?( \(?[a-z ]*\)?)?$|battery (capacity|size|energy|pack)|energy capacity|kwh', 'f', 'hi'),
 ('fastchg', 'Fast charging', C_PR, r'fast ?charg', 't', None),
 ('ct', 'Charging time (hrs)', C_PR, r'charging time|charge time|time to charge|full charge|charging duration', 'tm', 'lo'),
 ('chgkw', 'Charger (kW)', C_PR, r'charger|charging (power|rating|capacity)', 'pw', 'hi', 'charger'),
 ('mileage', 'Mileage (km/l or km/kg)', C_PR, r'mileage|fuel efficiency|km ?/ ?(l|kg)', 'f', 'hi'),
 # --- engine & motor
 ('nm', 'Torque (Nm)', C_EN, r'torque', 'nm', 'hi'),
 ('kwpk', 'Peak power (kW)', C_EN, r'peak (motor )?(power|output)|peak kw', 'pw', 'hi'),
 ('kw', 'Power (kW)', C_EN, r'power(?! ?steering| ?window| ?train)|\bkw\b|\bbhp\b|\bhp\b|\bps\b|horse', 'pw', 'hi'),
 ('cc', 'Displacement (cc)', C_EN, r'displacement|engine capacity|cubic|\bcc\b', 'cc', None),
 ('valves', 'Valves per cylinder', C_EN, r'valve', 'f', None),
 ('cyl', 'Cylinders', C_EN, r'cylinder', 't', None),
 ('bore', 'Bore x stroke (mm)', C_EN, r'bore', 't', None),
 ('comp', 'Compression ratio', C_EN, r'compression', 't', None),
 ('cool', 'Cooling', C_EN, r'cooling', 't', None),
 ('start', 'Starting system', C_EN, r'starting|starter|start system', 't', None),
 ('fuelsys', 'Fuel system', C_EN, r'fuel (system|injection|supply)|induction', 't', None),
 ('oilf', 'Oil filtration', C_EN, r'oil filt', 't', None),
 ('airf', 'Air filter', C_EN, r'air (filter|cleaner)', 't', None),
 ('tb', 'Throttle body size', C_EN, r'throttle', 't', None),
 ('counter', 'Counter balancer', C_EN, r'counter ?balanc', 't', None),
 ('clutch', 'Clutch', C_EN, r'clutch', 't', None),
 ('tank', 'Fuel tank (L)', C_EN, r'tank', 'tank', None),
 ('motor', 'Motor type', C_EN, r'^motor( type)?$|type of motor|motor (type|make)', 't', None),
 ('ctrl', 'Controller', C_EN, r'controller', 't', None),
 ('trans', 'Transmission / gears', C_EN, r'transmission|gear ?box|\bgears?\b|drive ?train|differential', 't', None),
 ('enginetype', 'Engine type', C_EN, r'engine type|^engine$', 't', None),
]
class Spec:
    def __init__(s, key, label, cat, rx, kind, dr=None, alt=None):
        s.key, s.label, s.cat, s.rx, s.kind, s.dir, s.alt = key, label, cat, re.compile(rx, re.I), kind, dr, alt
CANON = [Spec(*r) for r in CANON_ROWS]
SPEC = {s.key: s for s in CANON}
CAT_ORDER = [C_PR, C_EN, C_CP, C_EA, C_DM, C_CH, C_FT]
LAST_CATS = [C_DF, C_WR]

def match_key(label, ctx=''):
    for s in CANON:
        if s.rx.search(label): return s
    if ctx:
        c = ctx + ' ' + label
        for s in CANON:
            if s.rx.search(c): return s
    return None

# ------------------------------------------------------------------ brands
BRANDS = {'bajaj': 'Bajaj', 'bal': 'Bajaj', 'bajaj auto': 'Bajaj', 'tvs': 'TVS', 'tvs motor': 'TVS', 'mahindra': 'Mahindra', 'm&m': 'Mahindra',
          'mahindra & mahindra': 'Mahindra', 'piaggio': 'Piaggio', 'atul': 'Atul', 'atul auto': 'Atul', 'euler': 'Euler', 'tata': 'Tata',
          'tata motors': 'Tata', 'omega seiki': 'Omega Seiki', 'montra': 'Montra', 'altigreen': 'Altigreen', 'yc electric': 'YC Electric',
          'mayuri': 'Mayuri', 'yaatri': 'Yaatri', 'saarthi': 'Saarthi', 'jp': 'JP', 'kyoro': 'Kyoro', 'bemac': 'BEMAC', 'selis': 'Selis',
          'yadea': 'Yadea', 'elektroteq': 'Elektroteq', 'hermi': 'Hermi', 'youdha': 'Youdha', 'lohia': 'Lohia', 'kinetic green': 'Kinetic Green',
          'greaves': 'Greaves', 'ashok leyland': 'Ashok Leyland', 'eicher': 'Eicher', 'force': 'Force', 'ola': 'Ola'}
BRANDS.update({k.lower(): v for k, v in EXTRA_BRANDS.items()})

def brand_from_text(t):
    if not t: return None
    low = t.lower(); best = None
    for a, c in BRANDS.items():
        if re.search(r'(?<![a-z0-9])' + re.escape(a) + r'(?![a-z0-9])', low):
            if best is None or len(a) > len(best[0]): best = (a, c)
    return best[1] if best else None

def brand_exact(t):
    return BRANDS.get((t or '').lower().strip())

# ------------------------------------------------------------------ grid reading
def read_grid(ws):
    if not hasattr(ws, 'iter_rows'): return None
    mr_, mc_ = min(ws.max_row or 0, MAX_ROWS), min(ws.max_column or 0, MAX_COLS)
    if mr_ < 2 or mc_ < 2: return None
    g = [list(r) for r in ws.iter_rows(min_row=1, max_row=mr_, max_col=mc_, values_only=True)]
    for m in ws.merged_cells.ranges:                       # un-merge: copy the value into every cell of the range
        if m.min_row > mr_ or m.min_col > mc_: continue
        v = g[m.min_row - 1][m.min_col - 1]
        for r in range(m.min_row - 1, min(m.max_row, mr_)):
            for c in range(m.min_col - 1, min(m.max_col, mc_)):
                g[r][c] = v
    G = [[S(v) for v in row] + [None] * (mc_ - len(row)) for row in g]
    while G and not any(G[-1]): G.pop()
    if not G: return None
    nc = max((max((c for c, v in enumerate(row) if v), default=-1) for row in G), default=-1) + 1
    return [row[:nc] for row in G] if nc >= 2 else None

def num_like(s): return re.fullmatch(r'[\d.,\s%:/+\-]+', s) is not None
def colvals(G, c): return [row[c] for row in G if c < len(row) and row[c]]
def transpose(G):
    nc = max(len(r) for r in G)
    return [[(G[r][c] if c < len(G[r]) else None) for r in range(len(G))] for c in range(nc)]

def label_hits(vals): return sum(1 for v in vals if v and not num_like(v) and match_key(v.lower()))

def orientation(G):
    nr, nc = len(G), len(G[0])
    colscore = max((label_hits(colvals(G, c)) for c in range(min(nc, 5))), default=0)
    rowscore = max((label_hits(G[r]) for r in range(min(nr, 15))), default=0)
    return rowscore >= 3 and rowscore > colscore * 1.5

HEADER_WORDS = re.compile(r'^(parameters?|specifications?|specs?|models?|vehicles?|variants?|description|features?|items?|particulars|attributes?|s\.? ?no\.?|sr\.? ?no\.?)$', re.I)
SKIP_HEADER = re.compile(r'^(units?|uom|remarks?|notes?|source|sr\.? ?no\.?|s\.? ?no\.?|sl\.? ?no\.?|#|comments?|parameters?|specifications?|description|features?|category|particulars)$', re.I)
UNIT_RX = re.compile(r'^(kg|mm|cm|m|km|kw|nm|kwh|rs|inr|%|hrs?|h|l|ltr|cc|ps|hp|km/h|kmph|km/hr|v|ah|deg|°|lakh|no\.?|nos|min|sec|s|km/l|km/kg|w|ft|cu ?ft|kg/m|n\.m)$', re.I)
BRAND_LABEL = re.compile(r'^(brand|make|maker|oem|manufacturer|company|oem name)$', re.I)

def find_blocks(G):
    """Return list of dicts: label col, optional category col, vehicle cols."""
    nr, nc = len(G), len(G[0])
    cand = []
    for c in range(min(nc, 6)):
        vals = colvals(G, c)
        if len(vals) < 3: continue
        if sum(map(num_like, vals)) / len(vals) > 0.6: continue
        cand.append((len({v.lower() for v in vals}) + 2 * label_hits(vals), -c, c))
    if not cand: return []
    l0 = max(cand)[2]
    L0 = {v.lower() for v in colvals(G, l0)}
    labs = [l0]
    for c in range(l0 + 1, nc):                             # repeated label columns => several spec blocks side by side
        vals = colvals(G, c)
        if len(vals) >= 5 and sum(1 for v in vals if v.lower() in L0) / len(vals) >= 0.5: labs.append(c)
    def is_cat(c, l):
        if c < 0: return False
        v = colvals(G, c)
        if not v or sum(map(num_like, v)) / len(v) >= 0.5: return False
        return len({x.lower() for x in v}) <= 0.5 * max(1, len(colvals(G, l)))
    cats = []
    for i, l in enumerate(labs):
        c = l - 1
        cats.append(c if (i == 0 or c > labs[i - 1]) and is_cat(c, l) else None)
    blocks = []
    for i, l in enumerate(labs):
        end = (cats[i + 1] if cats[i + 1] is not None else labs[i + 1]) if i + 1 < len(labs) else nc
        vcols = [c for c in range(l + 1, end) if len(colvals(G, c)) >= 3]
        vals_u = [c for c in vcols if len(colvals(G, c)) >= 3 and sum(1 for v in colvals(G, c) if len(v) <= 8 and UNIT_RX.match(v)) / len(colvals(G, c)) >= 0.6]
        vcols = [c for c in vcols if c not in vals_u]
        if vcols: blocks.append(dict(l=l, cat=cats[i], v=vcols))
    return blocks

def find_header(G, blk, override=None):
    l, vcols = blk['l'], blk['v']
    if override: return override - 1
    best, bsc = None, -1
    for r in range(min(len(G), 20)):
        cells = [G[r][c] for c in vcols]
        texts = [x for x in cells if x and not num_like(x)]
        if len(texts) < max(1, 0.5 * len(vcols)): continue
        if sum(1 for x in cells if x and num_like(x)) > 0.3 * len(vcols): continue
        lab = G[r][l]
        sc = len({t.lower() for t in texts})
        if not lab or HEADER_WORDS.match(lab): sc += 2
        elif lab and match_key(lab.lower()): sc -= 3
        if sc > bsc: best, bsc = r, sc
    return best

# ------------------------------------------------------------------ row -> fields
def slug(s): return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')[:60]
SIMPLE = re.compile(r'[\d.,]+\s*[A-Za-z/%°.\-²³ ]{0,10}')
DYN = OrderedDict()          # key -> (display label, section, direction)

def add_text(out, key, text, prefix=''):
    t = prefix + text
    cur = out.get(key)
    if cur is None: out[key] = t
    elif isinstance(cur, str) and t.lower() not in cur.lower(): out[key] = cur + ' / ' + t

def build_fields(rows):
    out, raw, labels = {}, {}, []
    for sec, cat, label, text in rows:
        ll = label.lower(); labels.append(ll)
        spec = match_key(ll, (cat or sec or '').lower())
        if spec is None:
            disp = label if not cat or cat.lower() in ll else f"{cat} – {label}"
            key = 'x_' + slug(disp)
            dr = next((d for rx, d in DYNAMIC_DIRECTION if re.search(rx, disp, re.I)), None)
            if key not in DYN: DYN[key] = (disp, cat or sec or 'Additional parameters', dr)
            if dr and key not in out and f(text) is not None:
                out[key] = f(text)
                if not SIMPLE.fullmatch(text) and len(text) <= 60: raw[key] = text
            else:
                add_text(out, key, text)
            continue
        key = spec.key
        pre = ('F: ' if 'front' in ll else 'R: ' if 'rear' in ll else '') if key in ('brk', 'tyre') else ''
        if spec.kind in ('t', 'wty'):
            add_text(out, key, text, pre)
            if spec.kind == 'wty' and 'wkm' not in out and wk(text) is not None: out['wkm'] = wk(text)
            continue
        n = PARSE[spec.kind](text, label)
        if n is None:
            add_text(out, spec.alt or key, text, pre); continue
        if key in out and isinstance(out[key], (int, float)): continue
        out[key] = n
        if spec.kind != 'price' and not SIMPLE.fullmatch(text) and len(text) <= 60 and not (spec.kind == 'pw' and re.search(r'\b(ps|hp)\b', text, re.I)):
            raw[key] = text
        if spec.kind == 'gr' and re.search(r'deg|°', text, re.I) and '%' not in text: raw[key] = f"{text} (~{n:g}%)"
        if spec.alt and re.search(r'[a-z]{3,}', re.sub(r'kw|kwh', '', text, flags=re.I), re.I): add_text(out, spec.alt, text)
    if all(k in out and isinstance(out[k], (int, float)) for k in ('len', 'wid', 'hei')) and 'dim' not in out:
        out['dim'] = f"{out['len']:g} x {out['wid']:g} x {out['hei']:g}"
    for k in ('len', 'wid', 'hei'): out.pop(k, None)
    return out, raw, labels

# ------------------------------------------------------------------ fuel / use inference
FUELS = [('CNG', r'\bcng\b'), ('LPG', r'\blpg\b'), ('Diesel', r'diesel|\bdsl\b|oil[- ]cooled'), ('Petrol', r'petrol|gasoline')]
def fuel_from(text, ev=True):
    t = (text or '').lower()
    hits = [n for n, rx in FUELS if re.search(rx, t)]
    if hits: return hits[0]
    if ev and re.search(EV_NAME_HINTS, t): return 'EV'
    if re.search(r'(^|\W)(ice)(\W|$)', t): return 'ICE'
    return None

CARGO_RX = r'cargo|loader|pick-?up|\bdv\b|delivery|e-?cart|truck|kargo|goods|tipper|\bxtra\b|\bace\b|\bzor\b|\bload\b'
PASS_RX = r'passenger|e-?rick|rickshaw|\brik\b|\bauto\b|\bpaxx\b|x ?wide|\bcity\b|treo|\bre\b|compact'
def use_from(text):
    t = (text or '').lower()
    if re.search(CARGO_RX, t): return 'Cargo'
    if re.search(PASS_RX, t): return 'Passenger'
    return None

def infer(name, sheet, fname, out, labels):
    full = f"{fname} | {sheet} | {name}"
    fuel = fuel_from(name)
    if fuel is None:
        sf = fuel_from(sheet)
        if sf and sf != 'ICE': fuel = sf
    if fuel is None:
        ls = {n for n, rx in FUELS for l in labels if re.search(rx, l)}
        if ls: fuel = next(n for n, _ in FUELS if n in ls)
    if fuel is None:
        if any(k in out for k in ('bat', 'rng', 'ct', 'chgkw', 'batt', 'volt', 'regen')): fuel = 'EV'
        elif any(k in out for k in ('cc', 'tank', 'mileage', 'valves', 'comp', 'cyl')): fuel = 'ICE'
    if fuel is None: fuel = fuel_from(sheet) or fuel_from(fname) or 'ICE'
    use = None
    for rx, u in USE_OVERRIDES:
        if re.search(rx, full, re.I): use = u; break
    use = use or use_from(name) or use_from(sheet)
    if use is None:
        if any(k in out for k in ('seat', 'pseat', 'luggage', 'backrest')): use = 'Passenger'
        elif any(k in out for k in ('pay', 'tray', 'cft')): use = 'Cargo'
    use = use or use_from(fname) or 'Other'
    return fuel, use

# ------------------------------------------------------------------ sheet -> tables
def analyse(G, fname, sheet, report):
    out = []
    if orientation(G): G = transpose(G); report.append(f"    (sheet is laid out vehicles-in-rows: transposed automatically)")
    blocks = find_blocks(G)
    if not blocks: report.append("    no vehicle table detected"); return out
    for bi, blk in enumerate(blocks):
        l, vcols, catc = blk['l'], blk['v'], blk['cat']
        h = find_header(G, blk, HEADER_ROW_OVERRIDES.get((fname, sheet)))
        if h is None: continue
        # a header row made only of brand names -> real model row is the next row
        brandrow = {}
        hv = [G[h][c] for c in vcols if G[h][c]]
        if hv and sum(1 for x in hv if brand_exact(x)) / len(hv) >= 0.7 and h + 1 < len(G) and any(G[h + 1][c] for c in vcols):
            brandrow = {c: brand_exact(G[h][c]) for c in vcols}; h += 1
        brand_rows = set()
        for r in range(len(G)):
            if r == h: continue
            lab = G[r][l]; cells = [G[r][c] for c in vcols]
            if lab and BRAND_LABEL.match(lab) and any(cells):
                for c in vcols:
                    if G[r][c]:
                        b = brand_exact(G[r][c]) or G[r][c]; BRANDS.setdefault(G[r][c].lower(), b); brandrow.setdefault(c, b)
                brand_rows.add(r)
            elif not lab and r < h and sum(1 for x in cells if x) >= 1 and sum(1 for x in cells if x and brand_from_text(x)) >= 0.5 * sum(1 for x in cells if x):
                for c in vcols:
                    if G[r][c]: brandrow.setdefault(c, brand_from_text(G[r][c]))
                brand_rows.add(r)
        rows_by = {c: [] for c in vcols}
        cur_sec = None; prev_cat = None
        for r in range(len(G)):
            if r == h or r in brand_rows: continue
            lab = G[r][l]; cat = G[r][catc] if catc is not None else None
            if cat: prev_cat = cat
            if not lab:
                if cat and any(G[r][c] for c in vcols): lab, cat = cat, None
                else: continue
            vals = [G[r][c] for c in vcols]
            if not any(vals): cur_sec = lab; continue
            prev = None
            for c, v in zip(vcols, vals):
                if v and re.match(r'^(same( as above)?|as above|-?do-?|ditto|"|〃)$', v, re.I): v = prev
                if v: rows_by[c].append((cur_sec, (cat or prev_cat) if catc is not None else None, lab, v))
                prev = v
        nrows_all = len({x[2] for r in rows_by.values() for x in r})
        vehicles = []
        for c in vcols:
            name = G[h][c]
            if not name or num_like(name) or SKIP_HEADER.match(name): continue
            if EXCLUDE_VEHICLES and re.search(EXCLUDE_VEHICLES, name, re.I): continue
            brand = brandrow.get(c) or brand_from_text(name) or ''
            bj = brand == 'Bajaj' or bool(re.search(BAJAJ_NAME_HINTS, name, re.I))
            if bj: brand = 'Bajaj'
            fields, rawd, labels = build_fields(rows_by[c])
            if len(fields) < (1 if bj else min(MIN_FIELDS, max(1, nrows_all - 1))): continue
            fuel, use = infer(name, sheet, fname, fields, labels)
            vehicles.append(dict(n=name, b=brand, bj=1 if bj else 0, fuel=fuel, use=use, f=fields, raw=rawd, src=f"{os.path.splitext(fname)[0]} / {sheet}"))
        known = [v['use'] for v in vehicles if v['use'] != 'Other']
        if known:                                          # vehicles with no cargo/passenger clue take the sheet's majority
            top = max(set(known), key=known.count)
            for v in vehicles:
                if v['use'] == 'Other': v['use'] = top
        nrows = len({x[2] for r in rows_by.values() for x in r})
        report.append(f"    block {bi + 1}: header row {h + 1}, {len(vehicles)} vehicles ({sum(v['bj'] for v in vehicles)} Bajaj), {nrows} parameter rows")
        if len(vehicles) and nrows >= 2:
            out.append(dict(id=f"{fname}|{sheet}|{bi}", file=fname, sheet=sheet, vehicles=vehicles))
    return out

# ------------------------------------------------------------------ merge identical vehicles across sheets
def norm(s): return re.sub(r'[^a-z0-9]', '', (s or '').lower())
def key_name(v):
    n = v['n']
    if v['b']:
        n = re.sub(r'^\s*' + re.escape(v['b']) + r'\b[\s\-:]*', '', n, flags=re.I)
    return norm(n)
def fuel_compat(a, b): return a == b or 'ICE' in (a, b) and 'EV' not in (a, b)

def merge_tables(tables):
    recs = []; exact = {}
    tables = sorted(tables, key=lambda t: -sum(len(v['f']) for v in t['vehicles']) / len(t['vehicles']))
    for t in tables:
        t['members'] = []
        for v in t['vehicles']:
            k = (norm(v['b']), key_name(v), v['fuel'])
            r = exact.get(k)
            if r is None:
                for cand in recs:
                    if (cand['b'] == v['b'] or not cand['b'] or not v['b']) and fuel_compat(cand['fuel'], v['fuel']) \
                       and difflib.SequenceMatcher(None, key_name(cand), key_name(v)).ratio() >= 0.92:
                        r = cand; break
            if r is None:
                r = dict(v); r['id'] = f"v{len(recs)}"; r['fields'] = dict(v['f']); r['srcs'] = [v['src']]; r['ps'] = set()
                r.pop('f'); recs.append(r)
            else:
                for kk, vv in v['f'].items(): r['fields'].setdefault(kk, vv)
                for kk, vv in v['raw'].items(): r['raw'].setdefault(kk, vv)
                if v['src'] not in r['srcs']: r['srcs'].append(v['src'])
                if r['use'] == 'Other' and v['use'] != 'Other': r['use'] = v['use']
                if r['fuel'] == 'ICE' and v['fuel'] != 'ICE': r['fuel'] = v['fuel']
                if not r['b'] and v['b']: r['b'] = v['b']
                r['bj'] = max(r['bj'], v['bj'])
            exact[(norm(r['b']), key_name(r), r['fuel'])] = r
            t['members'].append(r)
    return recs, tables

def segment(r): return ('EV' if r['fuel'] == 'EV' else 'ICE') + ' ' + r['use']

def build_peer_sets(recs, tables, report):
    psets = []
    def mk(pid, label, members):
        psets.append(dict(id=pid, label=label, seg=segment(next(m for m in members if m['bj']))))
        for m in members: m['ps'].add(pid)
    orphans = []; members = {}
    for t in tables:
        B = [m for m in t['members'] if m['bj']]; Cm = [m for m in t['members'] if not m['bj']]
        label = f"{os.path.splitext(t['file'])[0]} / {t['sheet']}"
        if B and Cm:
            ids = {m['id'] for m in B + Cm}
            if any(ids <= mem for mem in members.values()): continue     # side sheet (e.g. earnings) of an existing peer set
            members[t['id']] = ids; mk(t['id'], label, B + Cm)
        elif Cm and not B: orphans.append((t, Cm, label))
    for t, Cm, label in orphans:
        ctx = f"{t['file']} | {t['sheet']}"; linked = []
        for rx, brx in LINKS.items():
            if re.search(rx, ctx, re.I): linked += [r for r in recs if r['bj'] and re.search(brx, r['n'], re.I)]
        if not linked:
            toks = {x.lower() for x in re.findall(r'[A-Za-z]*\d+[A-Za-z0-9]*', ctx) if len(x) >= 3}
            linked = [r for r in recs if r['bj'] and any(tk in norm(r['n']) for tk in toks)]
        if not linked:
            seg_rx = {segment(c) for c in Cm}
            linked = [r for r in recs if r['bj'] and segment(r) in seg_rx]
        if not linked:
            report.append(f"  ! competitor-only sheet '{ctx}' could not be linked to any Bajaj model (add it to LINKS)"); continue
        report.append(f"  competitor-only sheet '{ctx}' attached to: {', '.join(sorted({r['n'] for r in linked}))}")
        for b in linked:
            own = [p for p in psets if p['id'] in b['ps']]
            if own:
                for p in own:
                    for m in Cm: m['ps'].add(p['id'])
            else:
                mk(f"{t['id']}>{b['id']}", label, [b])
                for m in Cm: m['ps'].add(f"{t['id']}>{b['id']}")
    return psets

# ------------------------------------------------------------------ file discovery
def find_files(folder=None, depth=4):
    bases = [folder] if folder else [os.getcwd(), '/drive', '/content', '/mnt/data', '/mnt/user-data/uploads', os.path.expanduser('~')]
    found = {}
    for base in bases:
        if not base or not os.path.isdir(base): continue
        b0 = base.rstrip(os.sep).count(os.sep)
        for root, dirs, files in os.walk(base):
            if root.count(os.sep) - b0 >= depth: dirs[:] = []
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ('site-packages', 'node_modules', 'proc', 'sys', '__pycache__', 'outputs')]
            for fn in files:
                if not fn.lower().endswith(('.xlsx', '.xlsm')) or re.search(FILE_EXCLUDE, fn, re.I): continue
                p = os.path.join(root, fn)
                if fn not in found or os.path.getmtime(p) > os.path.getmtime(found[fn]): found[fn] = p   # newest copy wins
    return sorted(found.values())

def load_records(paths, report):
    import openpyxl
    tables = []
    for p in paths:
        fn = os.path.basename(p)
        try: wb = openpyxl.load_workbook(p, data_only=True)
        except Exception as e: report.append(f"! could not open {fn}: {e}"); continue
        report.append(f"{fn}")
        for ws in wb.worksheets:                           # every sheet, hidden ones included
            report.append(f"  sheet '{ws.title}'" + (" (hidden)" if ws.sheet_state != 'visible' else ''))
            G = read_grid(ws)
            if G is None: report.append("    empty / not a table"); continue
            try: tables += analyse(G, fn, ws.title, report)
            except Exception as e: report.append(f"    ! skipped ({type(e).__name__}: {e})")
    if not tables: return [], [], []
    recs, tables = merge_tables(tables)
    psets = build_peer_sets(recs, tables, report)
    return recs, psets, tables

def build_payload(recs, psets, paths):
    used = set(); out = []
    for r in recs:
        d = dict(id=r['id'], n=r['n'], b=r['b'], bj=r['bj'], fuel=r['fuel'], use=r['use'], ps=sorted(r['ps']), src=' ; '.join(r['srcs']))
        d.update(r['fields'])
        if 'wkm' in d and 'wty' not in d: d['wty'] = f"{d['wkm']:g} km"
        if r['raw']: d['raw'] = r['raw']
        out.append(d); used.update(k for k in d if k not in ('id', 'n', 'b', 'bj', 'fuel', 'use', 'ps', 'raw'))
    schema = []
    for cat in CAT_ORDER + LAST_CATS:
        rows = [[x.key, x.label, x.dir] if x.dir else [x.key, x.label] for x in CANON if x.cat == cat and x.key in used]
        if cat == C_DF: rows.append(['src', 'Source sheet(s)'])
        if cat == C_WR and 'wkm' in used: rows.insert(0, ['wkm', 'Warranty', 'hi'])
        schema.append([cat, rows])
    secs = OrderedDict()
    for k, (disp, sec, dr) in DYN.items():
        if k in used: secs.setdefault(sec.strip().title() if sec != 'Additional parameters' else sec, []).append([k, disp, dr] if dr else [k, disp])
    for sec, rows in secs.items(): schema.insert(len(schema) - 2, [sec, rows])
    schema = [x for x in schema if x[1]]
    return dict(rec=out, psets=psets, schema=schema, files=[os.path.basename(p) for p in paths], built=datetime.date.today().isoformat())

# ------------------------------------------------------------------ the dashboard page
PAGE = r"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Bajaj 3W Benchmarking</title>
<link href="https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
:root{--bg:#eef1f6;--pn:#fff;--ink:#0c1a2e;--mu:#62708a;--ln:#dde3ee;--bl:#1d5bff;--bls:#e8efff;--ev:#0e9f78;--evs:#e3f6ef;--ice:#e0861a;--ices:#fdf1de;--gn:#c9f0d3;--gt:#0a5c2b;--hero:#0b1f3a;--sh:0 1px 2px rgba(12,26,46,.06),0 8px 24px rgba(12,26,46,.06)}
@media (prefers-color-scheme:dark){:root{--bg:#0b121c;--pn:#141d2b;--ink:#e9eef7;--mu:#93a1b8;--ln:#26324a;--bl:#6c97ff;--bls:#1a2744;--evs:#12332b;--ices:#38280f;--gn:#1c4a2d;--gt:#a8efbf;--hero:#0a1730;--sh:none}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 Manrope,system-ui,-apple-system,"Segoe UI",Arial,sans-serif}
.hero{background:linear-gradient(120deg,var(--hero),#143a73);color:#fff;padding:34px 20px 74px}
.in{max-width:1240px;margin:0 auto}
.hero h1{margin:0 0 6px;font-size:30px;font-weight:800;letter-spacing:-.02em}
.hero p{margin:0;color:#b9c8e4;max-width:640px}
.stats{display:flex;gap:28px;margin-top:20px;flex-wrap:wrap}.stats b{display:block;font-size:26px;font-weight:800}.stats span{color:#9fb3d6;font-size:13px}
main{max-width:1240px;margin:-48px auto 0;padding:0 16px 56px}
.card{background:var(--pn);border-radius:16px;box-shadow:var(--sh);border:1px solid var(--ln);padding:20px;margin-bottom:18px}
h2{margin:0 0 4px;font-size:19px;font-weight:700}.hint{margin:0 0 16px;color:var(--mu);font-size:14px}
.tag{font-size:12px;font-weight:600;border-radius:999px;padding:3px 10px;background:var(--bg);color:var(--mu)}
.tag.ev{background:var(--evs);color:var(--ev)}.tag.ice{background:var(--ices);color:var(--ice)}.tag.bl{background:var(--bls);color:var(--bl)}
.score{min-width:260px;background:var(--bg);border-radius:12px;padding:12px 16px}
.score b{font-size:22px;font-weight:800}.score span{color:var(--mu);font-size:13px}
.bar{height:8px;border-radius:99px;background:var(--ln);margin-top:8px;overflow:hidden}.bar i{display:block;height:100%;background:var(--ev);border-radius:99px;transition:width .5s}
.opts{display:flex;gap:18px;flex-wrap:wrap;margin:6px 0 12px;color:var(--mu);font-size:13px}.opts label{display:flex;gap:6px;align-items:center;cursor:pointer}
.legend{display:flex;gap:8px;align-items:center;font-size:13px;color:var(--mu);margin-bottom:10px;flex-wrap:wrap}
.sw{display:inline-block;width:16px;height:16px;background:var(--gn);border:1px solid var(--gt);border-radius:4px}
.wrap{overflow-x:auto;border:1px solid var(--ln);border-radius:12px;max-width:100%}
table{border-collapse:separate;border-spacing:0;width:100%;font-size:13.5px}
th,td{padding:10px 14px;border-bottom:1px solid var(--ln);border-right:1px solid var(--ln);text-align:left;vertical-align:top;min-width:160px;max-width:240px;background:var(--pn)}
th:last-child,td:last-child{border-right:0}
thead th{position:sticky;top:0;z-index:2;font-weight:700}
thead th small{display:block;font-weight:600;color:var(--mu);font-size:12px}
thead th.bj{background:var(--bl);color:#fff}thead th.bj small{color:#d6e2ff}
.pl{position:sticky;left:0;z-index:1;min-width:180px;font-weight:600}thead .pl{z-index:3}
tr.gh td{background:var(--bg);color:var(--mu);font-weight:700;font-size:12.5px;position:static}
td.bj{background:var(--bls)}td.best{background:var(--gn);color:var(--gt);font-weight:700}td.best:before{content:"✓ "}
td.na{color:var(--mu);opacity:.5}.dir{color:var(--mu);font-weight:500;font-size:11px}
tfoot td{font-weight:800}
.note{color:var(--mu);font-size:12.5px;margin-top:14px;max-width:920px}
.fade{animation:f .35s ease}@keyframes f{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}
@media (prefers-reduced-motion:reduce){.fade{animation:none}.pc{transition:none}}
:focus-visible{outline:2px solid var(--bl);outline-offset:2px}
.ph{display:flex;align-items:center;justify-content:space-between;margin-bottom:8px}
.lnk{border:0;background:none;color:var(--bl);font:inherit;font-size:13px;font-weight:600;cursor:pointer;padding:2px 4px}
.pgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:20px}
.pchips{display:flex;flex-direction:column;gap:8px}
.pc{display:flex;align-items:center;gap:10px;text-align:left;border:2px solid var(--ln);background:var(--pn);color:var(--ink);border-radius:12px;padding:10px 12px;font:inherit;font-weight:600;cursor:pointer;transition:border-color .15s,background .15s}
.pc small{color:var(--mu);font-weight:600;margin-left:auto}
.pc:hover{border-color:var(--bl)}.pc.on{border-color:var(--bl);background:var(--bls)}
.ck{flex:none;width:20px;height:20px;border-radius:6px;border:2px solid var(--ln);display:grid;place-items:center;font-size:13px;color:#fff}
.pc.on .ck{background:var(--bl);border-color:var(--bl)}.pc.on .ck:after{content:"✓"}
.top{display:flex;justify-content:space-between;align-items:flex-start;gap:12px;flex-wrap:wrap;margin-bottom:14px}
.clr{border:1px solid var(--ln);background:var(--pn);color:var(--ink);border-radius:999px;padding:6px 14px;font:inherit;font-size:13px;font-weight:600;cursor:pointer}
.empty{text-align:center;color:var(--mu);padding:34px 10px}.empty b{display:block;color:var(--ink);font-size:18px;margin-bottom:4px}
.scores{display:flex;gap:10px;flex-wrap:wrap}.scores .score{min-width:220px}
.gtitle{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-bottom:12px}.gtitle h2{margin:0}
</style></head><body>
<div class="hero"><div class="in"><h1>Bajaj 3W benchmarking</h1><p>Select one or more Bajaj models and see every competitor in the dataset, with the best value in each parameter highlighted.</p><div class="stats" id="stats"></div></div></div>
<main>
<div class="card"><div class="top"><div><h2>Choose Bajaj models</h2><p class="hint" id="cnt" style="margin:0"></p></div><button class="clr" id="clr">Clear selection</button></div><div class="pgrid" id="pick"></div></div>
<div id="res"></div>
</main>
<script>
const DATA=__DATA__;
const D=DATA.rec,P=DATA.schema,PSM=Object.fromEntries(DATA.psets.map(p=>[p.id,p]));
const $=id=>document.getElementById(id);
const esc=s=>String(s).replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const rd=v=>Math.round(v).toLocaleString("en-IN");
const FM={price:v=>v.toFixed(2),earn:rd,opcost:rd,emi:rd,saving:rd,invest:rd,payback:v=>v.toFixed(1)};
const SEGORD=["EV Passenger","EV Cargo","ICE Passenger","ICE Cargo"];
const seg=r=>(r.fuel=="EV"?"EV":"ICE")+" "+r.use;
const sc_=s=>s.startsWith("EV")?"ev":s.startsWith("ICE")?"ice":"bl";
const so=s=>{const i=SEGORD.indexOf(s);return i<0?99:i};
const bj=D.filter(r=>r.bj);
const SEGS=[...new Set(bj.map(seg))].sort((a,b)=>so(a)-so(b)||a.localeCompare(b));
let sel=new Set(),hide=true;
$("stats").innerHTML=`<div><b>${bj.length}</b><span>Bajaj models</span></div><div><b>${D.length-bj.length}</b><span>competitor variants</span></div><div><b>${SEGS.length}</b><span>segments</span></div><div><b>${DATA.files.length}</b><span>workbooks read</span></div>`;
function fmt(r,k){const v=r[k];if(v==null)return null;
 if(k=="wkm")return r.wty||v.toLocaleString("en-IN")+" km";
 if(typeof v=="number")return (r.raw&&r.raw[k])||(FM[k]?FM[k](v):(Math.round(v*100)/100).toString());return v}
function picker(){$("pick").innerHTML=SEGS.map(s=>{const m=bj.filter(r=>seg(r)==s);
 return `<div><div class="ph"><span class="tag ${sc_(s)}">${esc(s)}</span><button class="lnk" data-all="${esc(s)}">Select all</button></div><div class="pchips">${m.map(r=>`<button class="pc ${sel.has(r.id)?"on":""}" data-id="${r.id}" aria-pressed="${sel.has(r.id)}"><span class="ck"></span>${esc(r.n)}${r.fuel&&r.fuel!="EV"?`<small>${esc(r.fuel)}</small>`:""}</button>`).join("")}</div></div>`}).join("");
 document.querySelectorAll(".pc").forEach(b=>b.onclick=()=>{const i=b.dataset.id;sel.has(i)?sel.delete(i):sel.add(i);b.classList.toggle("on");b.setAttribute("aria-pressed",sel.has(i));results()});
 document.querySelectorAll("[data-all]").forEach(b=>b.onclick=()=>{const m=bj.filter(r=>seg(r)==b.dataset.all);const all=m.every(r=>sel.has(r.id));m.forEach(r=>all?sel.delete(r.id):sel.add(r.id));picker();results()})}
function group(g,sels){const own=g.startsWith("own:"),comps=own?[]:D.filter(r=>!r.bj&&(r.ps||[]).includes(g)),cols=sels.concat(comps);
 const wins={};cols.forEach(c=>wins[c.id]=0);let rowsN=0,tb="";
 P.forEach(([gn,rows])=>{let body="";
  rows.forEach(([k,l,dir])=>{const vals=cols.map(c=>fmt(c,k));if(hide&&vals.every(v=>v==null))return;
   let best=null;if(dir){const n=cols.map(c=>typeof c[k]=="number"?c[k]:null).filter(v=>v!=null);if(n.length>1&&new Set(n).size>1){best=dir=="hi"?Math.max(...n):Math.min(...n);rowsN++}}
   body+=`<tr><td class="pl">${esc(l)}${dir?` <span class="dir">${dir=="hi"?"▲":"▼"}</span>`:""}</td>${cols.map((c,i)=>{const b=best!=null&&c[k]===best;if(b)wins[c.id]++;return `<td class="${b?"best":c.bj?"bj":""} ${vals[i]==null?"na":""}">${vals[i]==null?"–":esc(vals[i])}</td>`}).join("")}</tr>`});
  if(body)tb+=`<tr class="gh"><td colspan="${cols.length+1}">${esc(gn)}</td></tr>`+body});
 const sc=sels.map(s=>{const w=wins[s.id],pct=rowsN?Math.round(100*w/rowsN):0;return `<div class="score"><b>${w} of ${rowsN}</b> <span>${esc(s.n)} is best</span><div class="bar"><i style="width:${pct}%"></i></div></div>`}).join("");
 const sg=seg(sels[0]),lab=own?"Own specifications":(PSM[g]||{}).label||g;
 return `<div class="card fade"><div class="gtitle"><h2>${esc(sg)}</h2><span class="tag bl">${esc(lab)}</span><span class="tag">${sels.length} Bajaj · ${comps.length} competitors</span></div>
 <div class="scores" style="margin-bottom:14px">${sc}</div>${comps.length?"":`<p class="hint">No competitor data for this model in the uploaded files, so only its own specifications are shown.</p>`}
 <div class="legend"><span class="sw"></span>Best value for that parameter &nbsp;·&nbsp; ▲ higher is better &nbsp;·&nbsp; ▼ lower is better</div>
 <div class="wrap"><table><thead><tr><th class="pl">Parameter</th>${cols.map(c=>`<th class="${c.bj?"bj":""}"><small>${esc(c.b||"")} · ${esc(c.fuel)}</small>${esc(c.n)}</th>`).join("")}</tr></thead><tbody>${tb}</tbody>
 <tfoot><tr><td class="pl">Parameters where best</td>${cols.map(c=>`<td>${wins[c.id]}</td>`).join("")}</tr></tfoot></table></div></div>`}
function results(){$("cnt").textContent=sel.size?sel.size+" selected":"None selected yet. Tap any model.";
 const s=bj.filter(r=>sel.has(r.id));
 if(!s.length){$("res").innerHTML=`<div class="card empty"><b>No Bajaj model selected</b>Pick one or more models above and their competitors will appear here.</div>`;return}
 const m=new Map();s.forEach(r=>(r.ps&&r.ps.length?r.ps:["own:"+r.id]).forEach(p=>{if(!m.has(p))m.set(p,[]);m.get(p).push(r)}));
 const keys=[...m.keys()].sort((a,b)=>so(seg(m.get(a)[0]))-so(seg(m.get(b)[0]))||a.localeCompare(b));
 $("res").innerHTML=`<div class="opts"><label><input type="checkbox" id="hide" ${hide?"checked":""}> Hide parameters with no data</label></div>`+keys.map(k=>group(k,m.get(k))).join("")+
 `<p class="note">Built ${esc(DATA.built)} from: ${DATA.files.map(esc).join(", ")}. Every sheet (hidden ones included) is read automatically; models are compared only against competitors that share a sheet with them. Ranges such as "176 à 212" are compared on the first number. PS and HP converted to kW, kg·m to Nm, gradient degrees to %. Wheel-level torque is excluded. Parameters that are new in a workbook appear automatically under their sheet's section heading.</p>`;
 $("hide").onchange=e=>{hide=e.target.checked;results()}}
$("clr").onclick=()=>{sel.clear();picker();results()};
picker();results();
</script></body></html>"""

def run(folder=None, files=None, out_path="Bajaj_Benchmarking.html", height=1000, show=True, verbose=True):
    """folder: where the workbooks are (default: auto-search).  files: explicit list of workbook paths."""
    try: import openpyxl
    except ImportError:
        print("openpyxl is missing - run  %pip install openpyxl  and try again."); return None
    DYN.clear()
    paths = list(files) if files else find_files(folder)
    if not paths:
        print("No Excel workbooks found. Upload them next to the notebook (or call run('/path/to/folder'))."); return None
    report = []
    recs, psets, tables = load_records(paths, report)
    if verbose: print("\n".join(report))
    if not recs: print("\nNothing usable found in the workbooks."); return None
    data = build_payload(recs, psets, paths)
    page = PAGE.replace("__DATA__", json.dumps(data, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))
    nb = sum(r['bj'] for r in recs)
    try:
        with open(out_path, "w", encoding="utf-8") as fh: fh.write(page)
        print(f"\n{len(recs)} vehicles ({nb} Bajaj) from {len(paths)} workbook(s). Saved '{out_path}'.")
    except OSError:
        print(f"\n{len(recs)} vehicles ({nb} Bajaj) from {len(paths)} workbook(s).")
    if show:
        try:
            from IPython.display import HTML, display
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:1px solid #ccc"></iframe>' % (html.escape(page, quote=True), height)))
        except ImportError: pass
    return data

if __name__ == "__main__" or 'get_ipython' in globals():
    run()

In [3]:
"""@title
""""""Bajaj 3W benchmarking dashboard.

Reads the benchmarking workbooks, normalises every vehicle into one record, and writes a
self-contained interactive HTML dashboard (Bajaj_Benchmarking.html)."""
"""
import re, math, os, json, html, logging, warnings

log = logging.getLogger("bajaj_benchmarking")


# ============================================================ cell access & parsing helpers
def sheet(wb, name):
    for ws in wb.worksheets:
        if ws.title.strip().lower() == name.strip().lower(): return ws
    raise KeyError(f"Sheet '{name}' not found. Sheets: {[w.title for w in wb.worksheets]}")
def G(ws, coord):
    c = ws[coord]
    for m in ws.merged_cells.ranges:
        if coord in m: return ws.cell(m.min_row, m.min_col).value
    return c.value
def S(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).replace('\u200b', '').replace('\xa0', ' ')).strip()
    if not s or s.startswith('#') or s.lower() in ('-', '--', '---', 'na', 'n/a', 'nan', 'tbc', 'to be confirmed', 'to be checked', 'not provided'): return None
    return s
def f(s):
    s = S(s)
    if s is None: return None
    m = re.search(r'\d+(?:\.\d+)?', s.replace(',', '.') if re.match(r'^\d,\d', s) else s)
    return float(m.group()) if m else None
def pw(s, ps=False):
    s = S(s)
    if s is None: return None
    n = f(s.replace('9,1', '9.1'))
    if n is None: return None
    low = s.lower()
    if 'kw' in low: return n
    if 'ps' in low or ps: return round(n * 0.7355, 2)
    return n
def nm(s):
    s = S(s)
    if s is None: return None
    n = f(s)
    if n is None: return None
    if 'kg.m' in s.lower(): return round(n * 9.80665, 1)
    if 'wheel' in s.lower() or n > 300: return None
    return n
def gr(s):
    s = S(s)
    if s is None: return None
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', s)
    if m: return float(m.group(1))
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:deg|°)', s, re.I)
    if m: return round(math.tan(math.radians(float(m.group(1)))) * 100, 1)
    n = f(s)
    if n is None: return None
    return round(n * 100, 1) if n <= 1 else n
def gr_pct(s):  # sheets where "deg" is a mislabel for %
    s = S(s); n = f(s) if s else None
    return None if n is None else (round(n * 100, 1) if n <= 1 else n)
def tm(s):
    s = S(s)
    if s is None: return None
    m = re.search(r'(\d+)\s*:\s*(\d+)', s)
    if m: return round(int(m.group(1)) + int(m.group(2)) / 60, 2)
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:hrs?|hours?)\s*(?:(\d+)\s*min)?', s, re.I)
    if m: return round(float(m.group(1)) + (int(m.group(2)) / 60 if m.group(2) else 0), 2)
    return None
def wk(s):
    s = S(s)
    if s is None: return None
    s = re.sub(r'(?<=\d) (?=\d{3})', '', s).replace(',', '')
    m = re.search(r'(\d+(?:\.\d+)?)\s*(lakh|lac|l|k)?\s*kms?', s, re.I)
    if not m: return None
    n = float(m.group(1)); u = (m.group(2) or '').lower()
    return n * (100000 if u in ('lakh', 'lac', 'l') else 1000 if u == 'k' else 1)
def price(s):
    s = S(s)
    if s is None: return None
    n = f(s)
    return round(n / 1e5, 2) if n and n > 1000 else n
def cc(s):
    s = S(s)
    if s is None: return None
    m = re.search(r'(\d+(?:\.\d+)?)\s*(?:cc|cm)', s, re.I)
    if m: return float(m.group(1))
    return f(s) if re.match(r'^[\d.]+$', s) else None
def tank(s):
    s = S(s); m = re.search(r'(\d+(?:\.\d+)?)\s*L', s or ''); return float(m.group(1)) if m else None

NUMK = dict(price=price, rng=f, bat=f, ct=tm, kw=pw, kwps=lambda s: pw(s, True), nm=nm, pay=f, kerb=f, spd=f, grade=gr, gradep=gr_pct,
            gc=f, cc=cc, wb=f, tank=tank, chgkw=f, wkm=wk, cft=f, mileage=f, valves=f, earn=f, opcost=f, emi=f, saving=f, invest=f, payback=f, gvw=f, track=f)
RAWK = {'rng', 'ct', 'pay', 'grade', 'bat', 'spd'}


# ============================================================ record builder
class Builder:
    def __init__(self): self.R = []
    def add(self, id, n, b, bj, fuel, use, grp, ws, col, spec, inherit=None, override=None, minfields=4):
        out = {}
        for k, rows in spec.items():
            rows = rows if isinstance(rows, (list, tuple)) else [rows]
            vals = []
            for r in rows:
                v = S(G(ws, f"{col}{r}"))
                if v: vals.append(v)
            out[k] = ' / '.join(vals) if vals else None
        rec = dict(id=id, n=n, b=b, bj=1 if bj else 0, fuel=fuel, use=use, g=grp); raw = {}
        for k, txt in out.items():
            if txt is None: continue
            key = 'kw' if k == 'kwps' else 'wkm' if k == 'war' else 'grade' if k == 'gradep' else k
            if k in NUMK or k == 'war':
                num = (wk if k == 'war' else NUMK[k])(txt)
                if k == 'war': rec['wty'] = txt
                if num is None: continue
                rec[key] = num
                if fuel == 'EV' and key in RAWK and txt != f"{num:g}" and len(txt) < 38: raw[key] = txt
            else:
                rec[key] = txt
        if override: rec.update(override)
        if raw: rec['raw'] = raw
        if len([k for k in rec if k not in ('id','n','b','bj','fuel','use','g','raw')]) >= minfields or bj:
            self.R.append(rec)
        return rec


# ============================================================ workbook discovery
KEYS = dict(cargomodular='cargomodular', projectspec='projectspec', maxz='maxz', xwide='xwide', remodel='remodel', p5009='p5009benchmarking')
LABELS = dict(cargomodular="Cargo Modular benchmarking", projectspec="Project spec and features - Cargo", maxz="MAX Z",
              xwide="Benchmarking - X Wide", remodel="RE Model Spec & feature study", p5009="P5009 Benchmarking")

def find_files(extra_dirs=()):
    """"""Search common folders for the benchmarking workbooks (matched by file name). Returns {key: path or None}.""""""
    bases = [os.getcwd(), "/drive", os.path.expanduser("~"), "/content", "/mnt", "/home"] + list(extra_dirs)
    found = {k: None for k in KEYS}
    for base in bases:
        if not os.path.isdir(base): continue
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ('site-packages', 'node_modules', 'proc', 'sys')]
            for fn in files:
                if fn.startswith('~$') or not fn.lower().endswith(('.xlsx', '.xlsm')): continue
                norm = re.sub(r'[^a-z0-9]', '', fn.lower())
                for k, kw in KEYS.items():
                    if kw in norm and not found[k]: found[k] = os.path.join(root, fn)
    return found

def fill(rec, src, keys):
    for k in keys:
        if k not in rec and k in src: rec[k] = src[k]


# ============================================================ load every workbook into records
def load_records(paths):
    """"""paths = {key: file path or None}. Any missing workbook is skipped.""""""
    import openpyxl
    B = Builder(); add = B.add
    WB = lambda k: openpyxl.load_workbook(paths[k], data_only=True)
    has = lambda k: bool(paths.get(k))
    byid = lambda i: next((r for r in B.R if r['id'] == i), None)
    # ===== 1) Cargo Modular benchmarking: EV cargo peer set
    wsa = None
    if has('cargomodular'):
        wb = WB('cargomodular'); wsa = sheet(wb, 'Benchmarking')
        EVS = dict(price=34, rng=20, bat=9, ct=12, kw=7, nm=8, pay=24, kerb=23, spd=32, grade=22, gc=31, wb=30, cft=33, chgkw=11, war=26, charger=11, trans=10,
                   tyre=13, fs=14, rs=15, steer=28, brk=27, dim=29, tray=17, hvac=18, door=16, box=25, remarks=35)
        cols = [('T','Modular Cargo (Base)','Bajaj',1),('U','Modular Cargo (Premium, HVAC)','Bajaj',1),('V','Ace Pro EV','Tata',0),('W','Ace EV','Tata',0),('X','Zor Grand DV','Mahindra',0),
                ('Y','Zor Grand DV+','Mahindra',0),('Z','Zor Grand Pickup','Mahindra',0),('AA','Hi Load XR','Euler',0),('AB','Turbo EV 1000 City','Euler',0),('AC','Turbo EV 1000 Fast Charge','Euler',0),
                ('AD','Turbo EV 1000 Maxx','Euler',0),('AE','Maxima Cargo EV','Bajaj',1),('AH','Turbo King HD EV','TVS',0),('AJ','Ape ElectriK FX Max','Piaggio',0),('AL','E Loader','YC Electric',0),('AM','Rage+','Omega Seiki',0)]
        modb = None
        for col, n, b, bj in cols:
            rec = add('c1'+col, n, b, bj, 'EV', 'Cargo', 'evc', wsa, col, EVS)
            if col == 'T': modb = rec
            if col == 'U':
                for k in ('price','bat','kw','nm','pay','kerb','spd','gc','wb','chgkw','charger','trans','tyre','fs','rs','steer','dim','tray','wty','wkm'): fill(rec, modb, [k])
                rec['hvac'] = 'HVAC'
    # ===== 2) Project spec - Cargo: GoGo cargo, Maxima Cargo 260cc / 236cc
    if has('projectspec'):
        wb = WB('projectspec'); ws = sheet(wb, 'GOGO Cargo')
        GS = dict(price=4, dim=5, kerb=6, wb=7, gc=8, bat=9, rng=10, grade=11, spd=12, modes=13, kw=14, nm=15, trans=16, chgkw=17, charger=[17,18], ct=19, icl=20, revsens=21, usb=22, hill=23,
                  steer=24, shaft=25, tyre=26, tray=27, vol=28, docking=29, fascia=30, wheelcover=31, geofence=32, remimm=33, overspeed=34, limiter=35, shareloc=36, tele=37, lamp=38,
                  signature=40, rollover=41, hazard=42, diff=list(range(46, 55)))
        for col, n, b, bj in [('C','GoGo C9009','Bajaj',1),('D','GoGo C9012','Bajaj',1),('E','King Kargo HD','TVS',0),('F','Treo Zor Grand','Mahindra',0),('G','Ape E-Xtra','Piaggio',0),('H','HiLoad EV','Euler',0),('I','Rage+','Omega Seiki',0),('J','Altigreen neEV','Altigreen',0)]:
            add('c2'+col, n, b, bj, 'EV', 'Cargo', 'gogoc', ws, col, GS)
        PS = dict(price=5, dim=6, wb=7, cc=8, valves=9, kwps=10, nm=11, start=12, trans=13, cool=14, grade=15, tyre=17, gc=18, shaft=19, revlever=20, music=21, spare=22, docking=23, tray=24, diff=[27,28,29,30])
        IS = dict(dim=4, wb=5, cc=6, kw=7, nm=8, trans=9, grade=10, tank=11, tyre=13, gc=14, steer=15, shaft=16, revlever=17, music=18, docking=19, spare=20, tray=21, diff=[24,25,26])
        ws = sheet(wb, '260cc Cargo CNG')
        for col, n, b, bj in [('D','Maxima Cargo CNG 260cc LC - Reg. Tray','Bajaj',1),('E','Maxima Cargo CNG 260cc LC - Longer Tray','Bajaj',1),('F','King Kargo','TVS',0),('G','Ape Xtra LDX HT','Piaggio',0),('H','Gem Cargo Aqua 6F','Atul',0),('I','Alfa Plus','M&M',0)]:
            rec = add('c3'+col, n, b, bj, 'CNG', 'Cargo', 'icec260', ws, 'D' if col == 'E' else col, PS)
            if col in 'DE': rec['tray'] = S(G(ws, f"{col}24")); rec['dim'] = S(G(ws, f"{col}6")) or rec.get('dim')
        ws = sheet(wb, '236cc Cargo CNG')
        for col, n, b, bj in [('D','Maxima Cargo 236cc - Reg. Tray','Bajaj',1),('E','Maxima Cargo 236cc - Longer Tray','Bajaj',1),('F','Ape Xtra LDX HT','Piaggio',0),('G','Gem Cargo Aqua 6F','Atul',0),('H','Alfa Plus','M&M',0)]:
            rec = add('c4'+col, n, b, bj, 'CNG', 'Cargo', 'icec236', ws, 'D' if col == 'E' else col, IS)
            if col in 'DE': rec['tray'] = S(G(ws, f"{col}21")); rec['dim'] = S(G(ws, f"{col}4")) or rec.get('dim')
    # cross-file: extras for Maxima Cargo CNG (Cargo Modular sheet, col AF) + two ICE cargo competitors
    if wsa is not None and byid('c3E'):
        r = byid('c3E')
        for k, fn, row in [('price', price, 34), ('pay', f, 24), ('kerb', f, 23), ('spd', f, 32), ('tank', tank, 9)]:
            v = fn(G(wsa, f"AF{row}"))
            if v is not None: r[k] = v
        for col, n, b, fu in [('AI','Turbo King HD CNG','TVS','CNG'),('AK','Xtra Bada Diesel','Piaggio','Diesel')]:
            add('c1'+col, n, b, 0, fu, 'Cargo', 'icec260', wsa, col, dict(price=34, kw=7, nm=8, pay=24, kerb=23, spd=32, grade=22, gc=31, wb=30, dim=29, cc=7, tank=9, trans=10, tyre=13, fs=14, rs=15))
    # ===== 3) Benchmarking - X Wide: GoGo passenger, Xwide ICE, Riki
    if has('xwide'):
        wb = WB('xwide'); ws = sheet(wb, 'EV')
        XS = dict(dim=7, kerb=8, wb=9, gc=10, bat=11, rng=12, grade=13, spd=14, modes=[15,16], kw=17, nm=18, chgkw=19, charger=19, ct=20, icl=21, revsens=22, usb=23, hill=24, geofence=25, tele=31,
                  lamp=32, signature=33, rollover=35, hazard=36, shaft=37, tyre=38, spare=39, docking=42, fascia=43, wheelcover=44, pseat=45, luggage=46, chgalerts=30, cabin=34, grab=40,
                  hood=41, backrest=47, tcr=51, brk=52, fs=53, rs=54, steer=48)
        px = {}
        for col, n in [('D','GoGo P5009'),('E','GoGo P5012'),('F','GoGo P7012'),('I','GoGo P9018')]:
            px[col] = add('x3'+col, n, 'Bajaj', 1, 'EV', 'Passenger', {'D':'pev_re','E':'pev_re','F':'pev_z','I':'pev_x'}[col], ws, col, XS)
        fill(px['E'], px['D'], ['dim','kerb','wb','gc','tyre','icl','revsens','usb','hill','geofence','tele','lamp','signature','rollover','hazard','shaft','spare','docking','fascia','wheelcover','modes'])
        ws = sheet(wb, 'ICE')
        XI = dict(dim=6, wb=8, gc=9, cc=10, kwps=11, nm=12, trans=13, grade=14, tank=15, tyre=17, steer=18, shaft=19, brk=20, speedo=21, revlever=22, parking=23, handstart=24, music=25, jack=26,
                  docking=29, spare=30, limp=16, mudf=27, mudr=28, features=32, dealers=35)
        for col, n, b, bj, fu, g in [('I','Maxima Xwide Liquid Cooled','Bajaj',1,'CNG','picng'),('J','Maxima Xwide Air Cooled','Bajaj',1,'CNG','picng'),('K','Ape Auto Bada','Piaggio',0,'CNG','picng'),
                ('L','Gem Paxx Aqua','Atul',0,'CNG','picng'),('M','Alfa DX DUO','M&M',0,'CNG','picng'),('N','Maxima Xwide Oil Cooled','Bajaj',1,'Diesel','pidsl'),('O','Ape Auto Bada','Piaggio',0,'Diesel','pidsl'),
                ('P','Gem Paxx Diesel','Atul',0,'Diesel','pidsl'),('Q','Alfa DX Water Cooled','M&M',0,'Diesel','pidsl')]:
            add('x4'+col, n, b, bj, fu, 'Passenger', g, ws, col, XI)
        ws = sheet(wb, 'RIKI')
        RS = dict(dim=7, kerb=8, tray=9, tcr=10, step=12, corr=18, weather=[19,20], utility=21, tyre=11, trans=13, brk=[14,15], fs=16, rs=16, door=17, bat=22, rng=23, grade=29, spd=30, kw=31, charger=32, chgkw=32,
                  ct=33, icl=34, usb=35, hill=36, tele=37, signature=39, rollover=40, hazard=41)
        rb = add('x5C', 'Riki E-Rick (Base)', 'Bajaj', 1, 'EV', 'Passenger', 'riki_p', ws, 'C', RS)
        rp = add('x5D', 'Riki E-Rick (Premium)', 'Bajaj', 1, 'EV', 'Passenger', 'riki_p', ws, 'E', {k: v for k, v in RS.items() if k in ('kerb','bat','rng','grade')})
        d2 = add('x5Dd', 'tmp', 'Bajaj', 1, 'EV', 'Passenger', 'tmp', ws, 'D', {k: v for k, v in RS.items() if k in ('tyre','trans','ct','hazard')})
        B.R.remove(d2); rp.update({k: v for k, v in d2.items() if k not in ('id','n','b','bj','fuel','use','g')})
        fill(rp, rb, ['dim','tray','tcr','brk','fs','rs','door','spd','kw','charger','chgkw','icl','usb','hill','tele','signature','rollover'])
        rp['n'] = 'Riki E-Rick (Premium)'
        RC = dict(RS); RC['rng'] = 27
        add('x5F', 'Riki E-Cart', 'Bajaj', 1, 'EV', 'Cargo', 'riki_c', ws, 'F', RC)
        for rid, col in [('x5C', 'C'), ('x5D', 'E'), ('x5F', 'F')]:
            parts = [f"{S(G(ws, f'B{x}'))}: {S(G(ws, f'{col}{x}'))}" for x in range(23, 29) if S(G(ws, f'B{x}')) and S(G(ws, f'{col}{x}'))]
            if parts: byid(rid)['rngx'] = ' | '.join(parts)
    # ===== 4) MAX Z: Maxima Z (CNG/LPG), GoGo P7012 peers, Sheet6 competitor specs
    s6 = {}
    if has('maxz'):
        wb = WB('maxz'); ws = sheet(wb, 'Max Z EV benchmarking')
        ZE = dict(price=5, dim=7, kerb=8, wb=9, gc=10, kw=13, nm=14, spd=15, grade=16, bat=17, rng=18, chgkw=20, charger=20, ct=21, motor=23, trans=[24,25], tele=28, geofence=29, remimm=30,
                  overspeed=31, limiter=32, shareloc=33, icl=35, seat=36, hill=38, revsens=39, usb=40, modes=42, regen=43, brk=45, fs=46, rs=47, tyre=49, lamp=52, signature=53, rollover=55,
                  hazard=56, shaft=58, luggage=60, wading=11, chgalerts=34, tubeless=37, parkassist=41, cabin=54, pseat=59, backrest=63, grab=65, hood=66, spare=64, docking=67, fascia=68,
                  wheelcover=69, war=70)
        for col, n, b, bj in [('E','King','TVS',0),('F','Montra ePl 2.0','Montra',0),('G','Treo Plus','Mahindra',0),('H','Ape E-City FX Max','Piaggio',0),('I','Atul RIK Twin','Atul',0),('J','Ape E-City Ultra','Piaggio',0)]:
            rec = add('z1'+col, n, b, bj, 'EV', 'Passenger', 'pev_z', ws, col, ZE)
            if col == 'E': rec['ct'] = 3.5; rec.setdefault('raw', {})['ct'] = '100% in 3 hr 30'
        p7 = byid('x3F') or add('z1D', 'GoGo P7012', 'Bajaj', 1, 'EV', 'Passenger', 'pev_z', ws, 'D', ZE)
        v = price(G(ws, 'D5'))
        if v is not None: p7['price'] = v
        p7.pop('war', None); wt = S(G(ws, 'D70'))
        if wt: p7['wty'] = wt; p7['wkm'] = wk(wt) or p7.get('wkm')
        p7['seat'] = S(G(ws, 'D36')); p7['regen'] = S(G(ws, 'D43')); p7['luggage'] = S(G(ws, 'D60'))
        ws = sheet(wb, 'Max ICE benchmarking')
        ZI = dict(dim=7, kerb=8, wb=9, gc=10, kw=13, nm=14, spd=15, grade=16, cc=17, mileage=18, tcr=20, cool=21, trans=[24], seat=36, hill=38, revlever=43, brk=45, fs=46, rs=47, parking=48, tyre=51,
                  steer=60, shaft=61, tank=62, spare=69, docking=72, cyl=19, enginetype=21, clutch=25, tubeless=37, adjseat=41, abs=49, limp=63)
        for col, n, b, bj, fu, g in [('D','Maxima Z (CNG)','Bajaj',1,'CNG','maxz_cng'),('E','Ape Metro','Piaggio',0,'CNG','maxz_cng'),('F','Atul Rik','Atul',0,'CNG','maxz_cng'),('G','King Duramax','TVS',0,'CNG','maxz_cng'),
                                     ('H','Maxima Z (LPG)','Bajaj',1,'LPG','maxz_lpg'),('I','Atul Rik','Atul',0,'LPG','maxz_lpg'),('J','Gem Paxx','Atul',0,'LPG','maxz_lpg')]:
            add('z2'+col, n, b, bj, fu, 'Passenger', g, ws, col, ZI, minfields=5)
        ws6 = sheet(wb, 'Sheet6')
        def block(catc, labc, specc):
            d = {}; cur = ''
            for r in range(3, ws6.max_row + 1):
                cat = S(G(ws6, f"{catc}{r}")) if catc else None
                lab = S(G(ws6, f"{labc}{r}")); spec = S(G(ws6, f"{specc}{r}"))
                if cat: cur = cat.lower()
                if spec is None:
                    if lab and not catc: cur = lab.lower()
                    continue
                if spec.lower().startswith('same as'): continue
                L_ = (lab or cat or '').lower()
                def put(k, v):
                    if v is not None and k not in d: d[k] = v
                if L_ == 'displacement': put('cc', cc(spec))
                elif L_ == 'type' and 'engine' in cur: put('cc', cc(spec))
                elif L_ == 'max power': put('kw', pw(spec))
                elif L_ == 'max torque': put('nm', nm(spec))
                elif L_ == 'transmission': put('trans', spec)
                elif L_ in ('max speed', 'maximum speed', 'top speed'): put('spd', f(spec))
                elif L_ in ('maximum gradeability', 'gradeability'): put('grade', gr(spec))
                elif L_ in ('fuel tank capacity', 'cng tank capacity') and 'petrol' not in spec.lower(): put('tank', tank(spec))
                elif L_ == 'wheelbase': put('wb', f(spec))
                elif L_ == 'overall length': put('len', f(spec))
                elif L_ == 'overall width': put('wid', f(spec))
                elif L_ == 'overall height': put('hei', f(spec))
                elif L_ in ('ground clearance', 'min ground clearance', 'minimum ground clearance'): put('gc', f(spec))
                elif L_ == 'kerb weight': put('kerb', f(spec))
                elif L_ in ('gvw', 'max gvw (kg)'): put('gvw', f(spec))
                elif L_ == 'seating capacity': put('seat', spec)
                elif L_ == 'cooling': put('cool', spec)
                elif L_ == 'warranty': put('wty', spec); put('wkm', wk(spec))
                elif L_.startswith('load body size'): put('tray', spec)
                elif L_ == 'tyres' or (L_ in ('size', 'front & rear') and 'tyre' in cur): put('tyre', spec)
                elif L_ in ('front & rear', 'type') and 'brake' in cur: put('brk', spec)
            if all(k in d for k in ('len', 'wid', 'hei')): d['dim'] = f"{d['len']:g} x {d['wid']:g} x {d['hei']:g}"
            for k in ('len', 'wid', 'hei'): d.pop(k, None)
            return d
        s6 = dict(ape_classic=block(None, 'B', 'C'), gem_diesel=block('D', 'E', 'F'), gem_cng=block('G', 'H', 'I'), alfa_dx=block('J', 'K', 'L'), alfa_plus=block('J', 'K', 'M'),
                  mahindra_dx=block('N', 'O', 'P'), tvs_dm=block('Q', 'R', 'S'))
        s6['alfa_plus'] = {**s6['alfa_dx'], **s6['alfa_plus']}
    # ===== 5) RE Model Spec: GoGo P5009/P5012 peers, RE compact CNG / LPG / petrol
    if has('remodel'):
        wb = WB('remodel'); ws = sheet(wb, 'EV')
        RE = dict(price=5, dim=6, kerb=7, wb=8, gc=9, bat=10, rng=12, gradep=13, spd=14, modes=15, kw=[16,17], nm=18, chgkw=20, charger=20, ct=21, icl=[22,32], revsens=23, usb=24, hill=25,
                  geofence=26, tyre=33, brk=34, fs=35, rs=36, war=37, volt=11, wading=19, extra=38)
        for col, n, b, bj in [('D','GoGo P5009','Bajaj',1),('E','GoGo P5012','Bajaj',1),('F','King EV Max','TVS',0),('G','Treo','Mahindra',0),('H','Treo Plus','Mahindra',0),('I','Ape E-City FX Max','Piaggio',0),('J','Ape E-City Ultra','Piaggio',0)]:
            rec = add('r1'+col, n, b, bj, 'EV', 'Passenger', 'pev_re', ws, col, RE)
            if col == 'F': rec['ct'] = 3.5; rec.setdefault('raw', {})['ct'] = '100% in ~3 hr 30'
        ergo = {'d1': 44, 'd2': 45, 'd3': 46, 'd7': 47, 'a4': 48, 'a6': [50, 51]}
        for col, rid in [('D', 'x3D' if byid('x3D') else 'r1D'), ('F', 'r1F'), ('G', 'r1G'), ('H', 'r1I')]:
            r = byid(rid)
            if not r: continue
            for k, rows in ergo.items():
                rows = rows if isinstance(rows, list) else [rows]
                vals = [S(G(ws, f"{col}{x}")) for x in rows]; vals = [v for v in vals if v]
                if vals: r[k] = ' / '.join(vals)
        if byid('x3D') and byid('x3E'):      # X Wide records are richer: keep them, borrow price / warranty from this sheet
            fill(byid('x3D'), byid('r1D'), ['price']); fill(byid('x3E'), byid('r1E'), ['wty', 'wkm'])
            B.R[:] = [r for r in B.R if r['id'] not in ('r1D', 'r1E')]
        ws = sheet(wb, 'RE CNG LC')
        LC = dict(cc=6, valves=7, kwps=8, nm=9, trans=[10, 13], start=11, cool=14, dim=16, wb=17, tank=18, gc=20, tyre=21, steer=22, shaft=23, revlever=24, handstart=25, docking=26, spare=27,
                  speedo=29, idss=30, faremeter=31, price=32, war=33, oilf=12, limp=19)
        for col, n, b, bj in [('D','RE Compact CNG - Current','Bajaj',1),('E','RE Compact CNG - New (LC)','Bajaj',1),('F','King Duramax Plus','TVS',0),('G','Alfa Dx Duo','M&M',0)]:
            add('r2'+col, n, b, bj, 'CNG', 'Passenger', 're_lc', ws, col, LC)
        for shn, fu, off in [('RE CNG Air cooled Specs', 'CNG', 0), ('RE LPG Air cooled Specs', 'LPG', 1), ('RE Petrol Air cooled Specs', 'Petrol', 1)]:
            ws = sheet(wb, shn)
            A = dict(cc=8+off, valves=10+off, kwps=11+off, nm=12+off, start=14+off, trans=[13+off, 20+off], cool=21+off, dim=23+off, wb=24+off, grade=28+off, tank=29+off, tyre=31+off, gc=32+off,
                     revlever=33+off, docking=34+off, parking=35+off, handstart=36+off, jack=37+off, spare=39+off, speedo=41+off, idss=42+off, faremeter=43+off, price=44+off, war=45+off,
                     bore=7+off, comp=9+off, counter=15+off, oilf=16+off, airf=17+off, fuelsys=18+off, tb=19+off, storage=38+off, limp=30+off)
            if fu != 'CNG': A.pop('tank')
            names = {c: f"{(S(G(ws, f'{c}{3+off}')) or '').strip()} {(S(G(ws, f'{c}{4+off}')) or '').strip()}".strip() for c in 'DEFGHI'}
            for col, bj in [('D',1),('E',1),('F',0),('G',0),('H',0),('I',0)]:
                nme = {'D': f'RE236 Existing - {fu}', 'E': f'RE236 4V New - {fu}'}.get(col, names[col])
                add(f'r3{fu}{col}', nme, 'Bajaj' if bj else (names[col].split()[0] if names[col] else ''), bj, fu, 'Passenger', f"re_{fu.lower()}", ws, col, A)
    # ===== 6) Sheets/sections that sit outside the main tables
    def newrec(id, n, b, bj, fu, use, g, **kw):
        r = dict(id=id, n=n, b=b, bj=1 if bj else 0, fuel=fu, use=use, g=g); r.update({k: v for k, v in kw.items() if v is not None}); B.R.append(r); return r
    if has('cargomodular'):
        wb = WB('cargomodular'); ws = sheet(wb, 'Sheet1'); maker = ''
        nm4 = {'D':'Turbo EV 1000','E':'Ace Pro EV','F':'Ace EV DV','G':'Ace EV','H':'Zeo V1 DV','I':'Zeo V2 DV'}
        for col in 'DEFGHIJK':
            maker = S(G(ws, f"{col}3")) or maker; name = S(G(ws, f"{col}4")) or ''; bj = maker.upper() == 'BAL'; ac = S(G(ws, f"{col}5"))
            if bj: continue   # BAL 4W DV (AC / Non AC) removed
            pr = S(G(ws, f"{col}10")); prn = price(pr.replace(',', '')) if pr else None
            brand = {'EULER': 'Euler', 'TATA': 'Tata', 'M&M': 'Mahindra'}.get(maker.upper(), maker)
            r = newrec('s1'+col, nm4.get(col, name), brand, 0, 'EV', 'Cargo', 'ev4w', price=prn, bat=f(G(ws, f"{col}6")), rng=f(G(ws, f"{col}7")),
                       pay=f(G(ws, f"{col}8")), cft=f(G(ws, f"{col}9")), hvac=ac)
        ws = sheet(wb, 'Earning potential')
        for row, rid in {4:'c1X', 5:'c1Y', 6:'c1Z', 7:'c1AA', 8:'c1AH', 9:'c1AJ', 10:'c1AL', 11:'c1AM', 12:'c1T', 13:'c1U', 14:'c1AE'}.items():
            r = byid(rid)
            if not r: continue
            for k, col in dict(earn='H', opcost='I', emi='J', saving='L', invest='M', payback='P').items():
                v = G(ws, f"{col}{row}")
                if isinstance(v, (int, float)): r[k] = round(float(v), 1)
            pr = G(ws, f"E{row}")
            if 'price' not in r and isinstance(pr, (int, float)): r['price'] = round(pr / 1e5, 2)
    # ===== 7) Riki competitors (E-Cart / E-Rick Base & Premium) from the Tech Specs and E-Rickshaw landscape slides
    if has('xwide'):
        RK = lambda i, n, b, use, g, **kw: newrec(i, n, b, 0, 'EV', use, g, **kw)
        mr = dict(spd=45, wb=2030, kw=3.5, bat=10.75, trans='Single speed', chgkw=3.0, tyre='4-12 (~120/80 12 inch)', brk='Drum (Hydraulic)',
                  wty='Battery 3 yrs / 60k km; controller & motor 1 yr', wkm=60000, grade=10, gc=200, rng=175, ct=4)
        mraw = {'kw': '3.5 (peak power)', 'chgkw': '~3.0', 'grade': '10-12', 'rng': '175-200'}
        # -- E-Cart peers (riki_c)
        RK('x7L', 'Mayuri Loader', 'Mayuri', 'Cargo', 'riki_c', dim='2580 x 970 x 1800', spd=19.5, wb=2030, kerb=322, pay=400, gvw=722, kw=1.2, motor='BLDC', bat=4, volt='48 V, 130 Ah',
           trans='Single speed', chgkw=1, charger='1 kW', tyre='3.75 x 12', brk='Drum (Mechanical)', tray='1220 x 915', wty='Vehicle 1 yr / Battery 1 yr', grade=12, gc=200, rng=100, ct=8)
        RK('x7M', 'Mayuri Rattan (L5) Cargo', 'Mayuri', 'Cargo', 'riki_c', **mr, dim='3620 x 1270 x 1720', kerb=557, pay=450, gvw=1000, tray='4 x 6 sq ft',
           raw={**mraw, 'kerb': '~557 (official data not available)'})
        # -- E-Rick Base / Premium peers (riki_p)
        RK('x7A', 'Mayuri Rattan (L5) Auto', 'Mayuri', 'Passenger', 'riki_p', **mr, dim='2880 x 1210 x 1860', kerb=557, seat='D+4', raw=mraw)
        RK('x7Y', 'Yaatri Super', 'Yaatri', 'Passenger', 'riki_p', dim='2855 x 1025 x 1720', kerb=315, spd=25, tcr='3.1 m', tyre='3.75-12"', step='330 (D) / 415 (P)',
           trans='Single speed', brk='Drum (Mechanical)', rs='Rigid axle; leaf spring', door='No', corr='ED + paint', weather='Open front end', utility='No', bat=5.3, rng=117,
           rngx='Range (D+1, IDC): 117 km (as per test) | Range (D+4, City): 64 km (as per test)', accel='2.7 s (0-10) / 8.7 s (0-20)', grade=11.6, kw=1, chgkw=1, charger='1 kW off-board',
           ct=6.5, icl='LCD', usb='No', hill='No', tele='No', lamp='1', signature='LED DRL', rollover='No', hazard='No',
           raw={'spd': '25 (29.7 as per test)', 'rng': '117 (as per test)', 'ct': '6 hr 30 min'})
        RK('x7S', 'Saarthi Plus', 'Saarthi', 'Passenger', 'riki_p', dim='2937 x 1030 x 1825', kerb=331, spd=25, tcr='3.2 m', tyre='3.75-12"', step='335 (D) / 400 (P)',
           trans='Single speed', brk='Drum (Mechanical)', rs='Rigid axle; leaf spring', door='No', corr='ED + paint', weather='Open front end', utility='No', bat=5.1, rng=111,
           rngx='Range (D+1, IDC): 111 km (as per test) | Range (D+4, City): 73 km (as per test)', accel='2.8 s (0-10) / 9.1 s (0-20)', grade=12.7, kw=1.1, chgkw=1.5, charger='1.5 kW off-board',
           ct=4.5, icl='Analog', usb='No', hill='No', tele='No', lamp='1', signature='LED DRL', rollover='No', hazard='No',
           raw={'spd': '25 (27.1 as per test)', 'rng': '111 (as per test)', 'ct': '4 hr 30 min'})
        RK('x7T', 'M&M Treo Yaari', 'Mahindra', 'Passenger', 'riki_p', dim='2769 x 995 x 1750', kerb=281, spd=25, tyre='3.75-12"', step='330',
           trans='Direct drive', brk='Drum (Mechanical)', rs='Rigid axle; leaf spring', door='No', corr='ED + paint', weather='Full front fascia', utility='Yes', bat=3.69, rng=76,
           rngx='Range (D+1, IDC): 76 km (as per test) | Range (D+4, City): 56 km (as per test)', accel='2.4 s (0-10) / 8.8 s (0-20)', grade=12.3, kw=1.95, chgkw=2, charger='2 kW off-board',
           ct=2.5, icl='LCD', usb='No', hill='No', tele='Optional', lamp='2', signature='No', rollover='No', hazard='Yes',
           raw={'spd': '25 (24.5 as per test)', 'rng': '76 (as per test)', 'ct': '2 hr 30 min'})
    # ===== 8) P5009 benchmarking workbook, 'Consolidated' sheet only: EV competitors of GoGo P5009 (peer set 'pev_re')
    if has('p5009'):
        wb = WB('p5009')
        _SKIP = re.compile(r'not (specified|available|confirmed|verified)|to confirm|^confirm\b|variant-dependent|unavailable$', re.I)
        _SIMPLE = re.compile(r'[\d.,]+\s*[A-Za-z/%°]*')
        _NUMS = ('rng', 'spd', 'ct', 'pay', 'kerb', 'gvw', 'wb', 'gc', 'bat')
        _NOTE = dict(grade='Gradeability', rng='Range', spd='Top speed')
        def _cl(v):                                   # cleaned text, or None for blanks / 'Not specified' style placeholders
            s = S(v)
            return None if s is None or _SKIP.search(s) else s
        def _nrm(s): return re.sub(r'[^a-z0-9]', '', str(s).lower())
        def _lab(ws, lc, vc, cat=None):
            """"""{normalised label: value} for a label | value sheet. A row with no label continues the label above it.""""""
            d, last = {}, None
            for r in range(1, ws.max_row + 1):
                l = S(G(ws, f"{lc}{r}")); v = G(ws, f"{vc}{r}")
                if isinstance(v, str):
                    v = re.sub(r'(^|; )\d\.\s+', r'\1', re.sub(r'\s*[\r\n]+\s*', '; ', v.strip()))
                v = S(v)
                if l:
                    last = _nrm(((S(G(ws, f"{cat}{r}")) or '') if cat else '') + l)
                    if v is not None: d.setdefault(last, v)
                elif v is not None and last in d:
                    d[last] += '; ' + v
            return d
        def _X(d, k): return _cl(d.get(k))
        def _pair(a, b):
            if a and b: return a if a.lower() == b.lower() else f"F: {a} / R: {b}"
            return a or b
        def _join(*xs, sep='; '): return sep.join(x for x in xs if x) or None
        def _note(label, v): return f"{label}: {v}" if v else None
        def _normdim(s):
            s = _cl(s)
            if s is None: return None
            s = re.sub(r'(?<=\d),(?=\d)', '', s)
            s = re.sub(r'\s*mm\b', '', s, flags=re.I).replace('Approx.', '~').replace('~ ', '~')
            return re.sub(r'\s*[×x*]\s*', ' x ', s).strip()
        def _dimtxt(*ps):
            ps = [_normdim(p) for p in ps]
            return ' x '.join(ps) if all(ps) else None
        def _nu(t):
            m = re.search(r'\d+(?:\.\d+)?', re.sub(r'(?<=\d),(?=\d)', '', t)); return float(m.group()) if m else None
        def _power(t):                                # kW / W / hp  ->  kW
            m = re.search(r'(\d+(?:\.\d+)?)(?:\s*-\s*\d+(?:\.\d+)?)?\s*(kw|hp|w)\b', re.sub(r'(?<=\d),(?=\d)', '', t), re.I)
            if not m: return None
            v, u = float(m.group(1)), m.group(2).lower()
            return round(v / 1000 if u == 'w' else v * 0.7457 if u == 'hp' else v, 2)
        def _conv(d):
            """"""text dict -> record fields (numbers parsed, original wording kept in 'raw' when it carries extra information)""""""
            out, raw, notes = {}, {}, []
            for k, t in d.items():
                t = _cl(t)
                if t is None: continue
                if k == 'wty':
                    out['wty'] = t; v = wk(t)
                    if v is not None: out['wkm'] = v
                    continue
                if k in _NUMS: v = _nu(t)
                elif k in ('kw', 'kwpk'): v = _power(t)
                elif k == 'grade': v = gr(t)
                else: out[k] = t; continue
                if v is None:
                    if k in _NOTE: notes.append(f"{_NOTE[k]}: {t}")
                    continue
                out[k] = v
                if len(t) <= 60 and not _SIMPLE.fullmatch(t) and not (k in ('kw', 'kwpk') and re.search(r'\bhp\b', t, re.I)): raw[k] = t
                if k == 'grade' and re.search(r'deg|°', t, re.I) and '%' not in t: raw[k] = f"{t} (~{v:g}%)"
            if notes: out['extra'] = _join(out.get('extra'), *notes)
            if raw: out['raw'] = raw
            return out
        def _mk(i, n, b, fu, use, d):
            c = _conv(d); raw = c.pop('raw', None)
            return newrec(i, n, b, 0, fu, use, 'pev_re', raw=raw, **c)
        def _sh(name):
            try: return sheet(wb, name)
            except KeyError: return None

        # ---- 'Consolidated': one column per competitor
        def _cons(d):
            X = lambda k: _X(d, k)
            cap = X('batterycapacity'); tel, gps = X('telematics'), X('gpstracking')
            return dict(seat=X('seatingcapacity'), dim=_dimtxt(d.get('length'), d.get('width'), d.get('height')), wb=X('wheelbase'), gc=X('groundclearance'),
                        kerb=X('kerbweight'), pay=X('payloadcapacity'), gvw=X('grossvehicleweightgvw'), motor=X('motortype'), kw=X('ratedpower'), kwpk=X('peakpower'),
                        ctrl=X('controller'), spd=X('topspeed'), accel=X('acceleration'), grade=X('gradeability'), rng=X('drivingrange'), batt=X('batterytype'),
                        bat=cap if cap and 'kwh' in cap.lower() else None,
                        volt=_join(f"{X('nominalvoltage')} nominal" if X('nominalvoltage') else None, f"{X('systemvoltage')} system" if X('systemvoltage') else None,
                                   cap if cap and 'kwh' not in cap.lower() else None, sep=' / '),
                        ct=X('chargingtime'), fastchg=X('fastcharging'), charger=X('chargerrating'), fs=X('frontsuspension'), rs=X('rearsuspension'),
                        brk=_pair(X('frontbrake'), X('rearbrake')), regen=X('regenerativebraking'), body=X('bodystructure'),
                        trans=_join(X('transmission'), X('differential')), icl=X('instrumentcluster'),
                        tele=f"{tel} (GPS: {gps})" if tel and gps and gps != tel else (tel or (f"GPS: {gps}" if gps else None)),
                        hazard=X('hazardswitch'), bhot=X('batteryoverheatalarm'), reserve=X('reserverange'), wty=X('vehiclewarranty'),
                        remarks=_note('Vehicle type', X('vehicletype')))
        ws = _sh('Consolidated')
        if ws is not None:
            hr = next((r for r in range(1, 12) if (S(G(ws, f"C{r}")) or '').lower() == 'parameter'), 4)
            BR = {'jp': 'JP', 'kyoro': 'Kyoro', 'bemac': 'BEMAC', 'selis': 'Selis', 'yadea': 'Yadea', 'etx': 'Elektroteq', 'hermi': 'Hermi', 'youdha': 'Youdha'}
            for ci in range(4, ws.max_column + 1):
                col = openpyxl.utils.get_column_letter(ci); name = S(G(ws, f"{col}{hr}"))
                if not name: continue
                d = _cons(_lab(ws, 'C', col))
                d['country'] = _cl(G(ws, f"{col}2"))
                _mk('p1' + col, name, BR.get(name.lower().split()[0].strip('+-'), name.split()[0]), 'EV', 'Passenger', d)
    def merge(rid, d):
        r = byid(rid)
        if r:
            for k, v in d.items():
                if k not in r: r[k] = v
    if s6:
        merge('r2F', s6['tvs_dm']); merge('r2G', s6['alfa_dx']); merge('x4M', s6['alfa_dx']); merge('x4L', s6['gem_cng']); merge('x4P', s6['gem_diesel'])
        merge('x4Q', s6['mahindra_dx']); merge('c3I', s6['alfa_plus']); merge('c4H', s6['alfa_plus'])
        if any(r['g'] == 'pidsl' for r in B.R): newrec('x6A', 'Ape Auto Classic', 'Piaggio', 0, 'Diesel', 'Passenger', 'pidsl', **s6['ape_classic'])
    for r in B.R:
        if 'diff' in r: r['diff'] = r['diff'].replace(' / ', '; ')
        if len(r.get('remarks', 'xxxx')) < 4: r.pop('remarks', None)
    return B.R


# ============================================================ dashboard page (HTML + CSS + JS)
PAGE = r""""""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Bajaj 3W Benchmarking</title>
<link href="https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>:root{
  --bg:#f3f6fb;--pn:#fff;--ink:#0b1b33;--mu:#64748b;--ln:#e2e8f2;
  --bl:#0057b8;--bls:#e6f0fc;--ev:#0b9a73;--evs:#e1f5ee;--ice:#d97706;--ices:#fdf0dc;
  --gn:#d4f3df;--gt:#0a5c2b;--hero:#071a35;--zebra:#f8fafd;
  --r:16px;--sh:0 1px 2px rgba(11,27,51,.05),0 10px 30px rgba(11,27,51,.07)}
@media (prefers-color-scheme:dark){:root{
  --bg:#0a111c;--pn:#121b2a;--ink:#e8eef8;--mu:#93a1b8;--ln:#24324a;
  --bl:#6aa0ff;--bls:#172646;--evs:#11302a;--ices:#36270f;--gn:#1b4a2d;--gt:#a8efbf;
  --hero:#081530;--zebra:#0f1826;--sh:none}}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;background:var(--bg);color:var(--ink);
  font:15px/1.55 Manrope,system-ui,-apple-system,"Segoe UI",Arial,sans-serif;-webkit-font-smoothing:antialiased}
.in{max-width:1280px;margin:0 auto}

/* hero */
.hero{position:relative;overflow:hidden;color:#fff;padding:44px 20px 96px;
  background:radial-gradient(900px 340px at 85% -10%,rgba(80,150,255,.35),transparent 60%),
             linear-gradient(125deg,var(--hero) 0%,#0b3a7a 100%)}
.hero:after{content:"";position:absolute;inset:auto -10% -60% -10%;height:100%;
  background:repeating-linear-gradient(115deg,rgba(255,255,255,.035) 0 2px,transparent 2px 26px);pointer-events:none}
.hero .in{position:relative;z-index:1}
.hero h1{margin:0 0 8px;font-size:clamp(26px,4vw,38px);font-weight:800;letter-spacing:-.03em;line-height:1.1}
.hero p{margin:0;color:#bcd0ee;max-width:620px;font-size:16px}
.stats{display:flex;gap:14px;margin-top:26px;flex-wrap:wrap}
.stats>div{background:rgba(255,255,255,.09);border:1px solid rgba(255,255,255,.14);
  backdrop-filter:blur(6px);border-radius:14px;padding:12px 20px;min-width:140px}
.stats b{display:block;font-size:28px;font-weight:800;letter-spacing:-.02em;line-height:1.15}
.stats span{color:#a9c0e6;font-size:12.5px;font-weight:600;text-transform:uppercase;letter-spacing:.06em}

/* layout + cards */
main{max-width:1280px;margin:-58px auto 0;padding:0 16px 64px;position:relative;z-index:2}
.card{background:var(--pn);border-radius:var(--r);box-shadow:var(--sh);border:1px solid var(--ln);padding:22px;margin-bottom:20px}
h2{margin:0 0 4px;font-size:19px;font-weight:800;letter-spacing:-.01em}
.hint{margin:0 0 16px;color:var(--mu);font-size:14px}

/* tags */
.tags{display:flex;gap:6px;flex-wrap:wrap}
.tag{font-size:12px;font-weight:700;border-radius:999px;padding:4px 11px;background:var(--bg);color:var(--mu);white-space:nowrap}
.tag.ev{background:var(--evs);color:var(--ev)}.tag.ice{background:var(--ices);color:var(--ice)}.tag.bl{background:var(--bls);color:var(--bl)}

/* picker */
.top{display:flex;justify-content:space-between;align-items:flex-start;gap:12px;flex-wrap:wrap;margin-bottom:18px}
.ph{display:flex;align-items:center;justify-content:space-between;margin-bottom:10px}
.lnk{border:0;background:none;color:var(--bl);font:inherit;font-size:13px;font-weight:700;cursor:pointer;padding:4px 6px;border-radius:8px}
.lnk:hover{background:var(--bls)}
.pgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:22px}
.pchips{display:flex;flex-direction:column;gap:8px}
.pc{display:flex;align-items:center;gap:10px;text-align:left;border:1.5px solid var(--ln);background:var(--pn);color:var(--ink);
  border-radius:12px;padding:10px 12px;font:inherit;font-size:14px;font-weight:600;cursor:pointer;
  transition:border-color .15s,background .15s,transform .15s,box-shadow .15s}
.pc small{color:var(--mu);font-weight:700;margin-left:auto;font-size:11.5px;text-transform:uppercase;letter-spacing:.04em}
.pc:hover{border-color:var(--bl);transform:translateY(-1px);box-shadow:0 4px 12px rgba(0,87,184,.12)}
.pc.on{border-color:var(--bl);background:var(--bls)}
.ck{flex:none;width:20px;height:20px;border-radius:6px;border:1.5px solid var(--ln);display:grid;place-items:center;font-size:12px;color:#fff;transition:all .15s}
.pc.on .ck{background:var(--bl);border-color:var(--bl)}.pc.on .ck:after{content:"✓"}
.clr{border:1px solid var(--ln);background:var(--pn);color:var(--ink);border-radius:999px;padding:7px 16px;font:inherit;font-size:13px;font-weight:700;cursor:pointer;transition:background .15s}
.clr:hover{background:var(--bg)}

/* results */
.gtitle{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-bottom:14px}.gtitle h2{margin:0}
.scores{display:grid;grid-template-columns:repeat(auto-fill,minmax(230px,1fr));gap:12px}
.score{background:var(--bg);border:1px solid var(--ln);border-radius:14px;padding:14px 16px}
.score b{font-size:22px;font-weight:800;letter-spacing:-.02em}.score span{color:var(--mu);font-size:13px}
.bar{height:7px;border-radius:99px;background:var(--ln);margin-top:10px;overflow:hidden}
.bar i{display:block;height:100%;background:linear-gradient(90deg,var(--ev),#34d3a6);border-radius:99px;transition:width .6s cubic-bezier(.2,.8,.2,1)}
.opts{display:flex;gap:18px;flex-wrap:wrap;margin:0 0 14px;color:var(--mu);font-size:13.5px;font-weight:600}
.opts label{display:flex;gap:8px;align-items:center;cursor:pointer}
.opts input{accent-color:var(--bl);width:16px;height:16px}
.legend{display:flex;gap:8px;align-items:center;font-size:12.5px;color:var(--mu);margin:14px 0 10px;flex-wrap:wrap}
.sw{display:inline-block;width:14px;height:14px;background:var(--gn);border:1px solid var(--gt);border-radius:4px}

/* table */
.wrap{overflow:auto;max-height:78vh;border:1px solid var(--ln);border-radius:12px;max-width:100%}
table{border-collapse:separate;border-spacing:0;width:100%;font-size:13.5px}
th,td{padding:10px 14px;border-bottom:1px solid var(--ln);border-right:1px solid var(--ln);text-align:left;vertical-align:top;
  min-width:160px;max-width:240px;background:var(--pn)}
th:last-child,td:last-child{border-right:0}
tbody tr:nth-child(even) td{background:var(--zebra)}
thead th{position:sticky;top:0;z-index:2;font-weight:800;font-size:13.5px;padding-top:12px;padding-bottom:12px}
thead th small{display:block;font-weight:700;color:var(--mu);font-size:11px;text-transform:uppercase;letter-spacing:.05em;margin-bottom:2px}
thead th.bj{background:linear-gradient(135deg,#0057b8,#1a7bff);color:#fff}
thead th.bj small{color:#cfe1ff}
.pl{position:sticky;left:0;z-index:1;min-width:190px;font-weight:700;border-right:2px solid var(--ln)}
thead .pl{z-index:3}
tr.gh td{background:var(--bls);color:var(--bl);font-weight:800;font-size:12px;text-transform:uppercase;letter-spacing:.07em;position:static;padding:8px 14px}
tbody tr:nth-child(even).gh td{background:var(--bls)}
tbody tr:hover td:not(.best):not(.bj){background:var(--bls)}
tbody td.bj{background:var(--bls)}
tbody td.best{background:var(--gn);color:var(--gt);font-weight:800}
td.best:before{content:"✓ "}
td.na{color:var(--mu);opacity:.45}
.dir{color:var(--mu);font-weight:600;font-size:10.5px}
tfoot td{font-weight:800;background:var(--bg);position:sticky;bottom:0;border-top:2px solid var(--ln)}

/* empty state + motion */
.empty{text-align:center;color:var(--mu);padding:48px 12px}
.empty b{display:block;color:var(--ink);font-size:19px;margin-bottom:6px}
.fade{animation:f .4s ease}@keyframes f{from{opacity:0;transform:translateY(10px)}to{opacity:1;transform:none}}
@media (prefers-reduced-motion:reduce){.fade{animation:none}.pc,.bar i{transition:none}}
:focus-visible{outline:2px solid var(--bl);outline-offset:2px}
@media (max-width:640px){.hero{padding:30px 16px 84px}th,td{padding:8px 10px;min-width:130px}.pl{min-width:140px}}</style></head><body>
<div class="hero"><div class="in"><h1>Bajaj 3W benchmarking</h1><p>Select one or more Bajaj models and see every competitor in the dataset, with the best value in each parameter highlighted.</p><div class="stats" id="stats"></div></div></div>
<main>
<div class="card"><div class="top"><div><h2>Choose Bajaj models</h2><p class="hint" id="cnt" style="margin:0"></p></div><button class="clr" id="clr">Clear selection</button></div><div class="pgrid" id="pick"></div></div>
<div id="res"></div>
</main>
<script>
const D=__DATA__;
const $=id=>document.getElementById(id);
const P=[["Price & range",[["price","Ex-showroom (Rs lakh)","lo"],["rng","Range (km)","hi"],["rngx","Range variants"],["bat","Battery (kWh)","hi"],["volt","Battery voltage"],["batt","Battery type"],["ct","Charging time (hrs)","lo"],["chgkw","Charger (kW)","hi"],["charger","Charger type"],["fastchg","Fast charging"],["reserve","Reserve range"],["mileage","Mileage (km/l or km/kg)","hi"]]],["Engine & motor",[["kw","Power (kW)","hi"],["kwpk","Peak power (kW)","hi"],["nm","Torque (Nm)","hi"],["cc","Displacement (cc)"],["valves","Valves per cylinder"],["cyl","Cylinders"],["bore","Bore x stroke (mm)"],["comp","Compression ratio"],["enginetype","Engine type"],["tank","Fuel tank (L)"],["cool","Cooling"],["start","Starting system"],["fuelsys","Fuel system"],["oilf","Oil filtration"],["airf","Air filter"],["tb","Throttle body size"],["counter","Counter balancer"],["clutch","Clutch"],["trans","Transmission / gears"],["motor","Motor type"],["ctrl","Controller"],["regen","Regenerative braking"]]],["Capacity & performance",[["pay","Payload (kg)","hi"],["kerb","Kerb weight (kg)","lo"],["gvw","GVW (kg)"],["spd","Top speed (km/h)","hi"],["grade","Gradeability (%)","hi"],["gc","Ground clearance (mm)","hi"],["cft","Cargo volume (cu ft)","hi"],["wading","Water wading"],["accel","Acceleration"]]],["Earning potential (35 km/day, Rs 3/kg freight)",[["earn","Earning per day (Rs)","hi"],["opcost","Daily operating cost (Rs)","lo"],["emi","EMI per day (Rs)","lo"],["saving","Saving per day (Rs)","hi"],["invest","Initial investment (Rs)","lo"],["payback","Payback (months)","lo"]]],["Dimensions & space",[["dim","Vehicle LxWxH (mm)"],["wb","Wheelbase (mm)"],["tray","Load tray (mm)"],["box","Cargo box (mm)"],["vol","Delivery van volume"],["luggage","Rear luggage space"],["seat","Seating"],["pseat","Passenger seat"],["backrest","Backrest to B-pillar (mm)"],["tcr","Turning radius"]]],["Comfort distances",[["d1","Scudo to battery cover, 10 cm above floor"],["d2","Leg room, dashboard to seat"],["d3","Scudo to battery cover, top edge"],["d7","Front leg room, 10 cm above floor"],["a4","Passenger door width"],["a6","Passenger legroom"]]],["Chassis",[["tyre","Tyre"],["brk","Brakes"],["fs","Front suspension"],["rs","Rear suspension"],["steer","Steering"],["body","Body / chassis"],["shaft","Drive shaft"],["door","Side door"],["hvac","HVAC / AC"],["spare","Spare wheel location"],["parking","Parking brake"],["limp","Limp-home tank"],["storage","Under-seat storage"],["mudf","Front mudguard"],["mudr","Rear mudguard"],["jack","Jack location"],["tubeless","Tubeless tyre"],["abs","ABS"],["adjseat","Adjustable driver seat"]]],["Features",[["modes","Drive modes"],["icl","Instrument cluster"],["speedo","Speedometer"],["hill","Hill hold"],["usb","USB charger"],["revsens","Reverse parking sensor"],["parkassist","Park assist"],["tele","Telematics"],["geofence","Geofencing / immobilisation"],["remimm","Remote immobilisation"],["overspeed","Overspeed alert"],["limiter","Speed limiter"],["shareloc","Share live location"],["chgalerts","Charging alerts"],["lamp","Lamps"],["signature","Signature lamp"],["cabin","Cabin light"],["rollover","Roll-over detection"],["hazard","Auto hazard"],["bhot","Battery overheat alarm"],["grab","Grab handle"],["hood","Hood frame"],["docking","Mobile docking"],["revlever","Reverse gear lever"],["handstart","Hand start lever"],["music","Music system provision"],["idss","IDSS"],["faremeter","Digital faremeter"],["fascia","Front fascia"],["wheelcover","Wheel cover"],["step","Step-in height (mm)"],["corr","Corrosion prevention"],["weather","Weather protection"],["utility","Utility space"],["extra","Other features"],["features","Feature notes"]]],["Differentiators & notes",[["country","Country supplied"],["diff","Differentiators"],["remarks","Remarks"],["dealers","Dealerships"]]],["Warranty",[["wkm","Warranty","hi"],["wty","Warranty terms"]]]];
const rd=v=>Math.round(v).toLocaleString("en-IN");const FM={price:v=>v.toFixed(2),earn:rd,opcost:rd,emi:rd,saving:rd,invest:rd,payback:v=>v.toFixed(1)};
const GL={"evc":["EV Cargo","Modular Cargo / Maxima Cargo EV peers"],"ev4w":["EV Cargo","4W delivery van (DV) peers"],"gogoc":["EV Cargo","GoGo cargo peers"],"riki_c":["EV Cargo","Riki E-Cart"],"pev_re":["EV Passenger","GoGo P5009 / P5012 peers"],"pev_z":["EV Passenger","GoGo P7012 peers"],"pev_x":["EV Passenger","GoGo P9018"],"riki_p":["EV Passenger","Riki E-Rick"],"icec260":["ICE Cargo","Maxima Cargo CNG 260cc peers"],"icec236":["ICE Cargo","Maxima Cargo 236cc peers"],"picng":["ICE Passenger","Maxima Xwide CNG peers"],"pidsl":["ICE Passenger","Maxima Xwide diesel peers"],"maxz_cng":["ICE Passenger","Maxima Z CNG peers"],"maxz_lpg":["ICE Passenger","Maxima Z LPG peers"],"re_lc":["ICE Passenger","RE Compact CNG (liquid-cooled) peers"],"re_cng":["ICE Passenger","RE236 CNG air-cooled peers"],"re_lpg":["ICE Passenger","RE236 LPG peers"],"re_petrol":["ICE Passenger","RE236 petrol peers"]};
const SEGS=["EV Passenger","EV Cargo","ICE Passenger","ICE Cargo"];
const seg=r=>(r.fuel=="EV"?"EV":"ICE")+" "+r.use;
const sc_=s=>s.startsWith("EV")?"ev":s.startsWith("ICE")?"ice":"bl";
const bj=D.filter(r=>r.bj);
let sel=new Set(),hide=true;
$("stats").innerHTML=`<div><b>${bj.length}</b><span>Bajaj models</span></div><div><b>${D.length-bj.length}</b><span>competitor variants</span></div><div><b>4</b><span>segments</span></div>`;
function fmt(r,k){const def=FM[k];const v=r[k];if(v==null)return null;
 if(k=="wkm")return r.wty||v.toLocaleString("en-IN")+" km";
 if(typeof v=="number")return (r.raw&&r.raw[k])||(def?def(v):(Math.round(v*100)/100).toString());return v}
function picker(){$("pick").innerHTML=SEGS.map(s=>{const m=bj.filter(r=>seg(r)==s);
 return `<div><div class="ph"><span class="tag ${sc_(s)}">${s}</span><button class="lnk" data-all="${s}">Select all</button></div><div class="pchips">${m.map(r=>`<button class="pc ${sel.has(r.id)?"on":""}" data-id="${r.id}" aria-pressed="${sel.has(r.id)}"><span class="ck"></span>${r.n}${r.fuel&&r.fuel!="EV"?`<small>${r.fuel}</small>`:""}</button>`).join("")}</div></div>`}).join("");
 document.querySelectorAll(".pc").forEach(b=>b.onclick=()=>{const i=b.dataset.id;sel.has(i)?sel.delete(i):sel.add(i);b.classList.toggle("on");b.setAttribute("aria-pressed",sel.has(i));results()});
 document.querySelectorAll("[data-all]").forEach(b=>b.onclick=()=>{const m=bj.filter(r=>seg(r)==b.dataset.all);const all=m.every(r=>sel.has(r.id));m.forEach(r=>all?sel.delete(r.id):sel.add(r.id));picker();results()})}
function group(g,sels){const comps=D.filter(r=>r.g==g&&!r.bj),cols=sels.concat(comps);
 const wins={};cols.forEach(c=>wins[c.id]=0);let rowsN=0,tb="";
 P.forEach(([gn,rows])=>{let body="";
  rows.forEach(([k,l,dir,df])=>{const vals=cols.map(c=>fmt(c,k,df));if(hide&&vals.every(v=>v==null))return;
   let best=null;if(dir){const n=cols.map(c=>typeof c[k]=="number"?c[k]:null).filter(v=>v!=null);if(n.length>1&&new Set(n).size>1){best=dir=="hi"?Math.max(...n):Math.min(...n);rowsN++}}
   body+=`<tr><td class="pl">${l}${dir?` <span class="dir">${dir=="hi"?"▲":"▼"}</span>`:""}</td>${cols.map((c,i)=>{const b=best!=null&&c[k]===best;if(b)wins[c.id]++;return `<td class="${b?"best":c.bj?"bj":""} ${vals[i]==null?"na":""}">${vals[i]==null?"–":vals[i]}</td>`}).join("")}</tr>`});
  if(body)tb+=`<tr class="gh"><td colspan="${cols.length+1}">${gn}</td></tr>`+body});
 const sc=sels.map(s=>{const w=wins[s.id],pct=rowsN?Math.round(100*w/rowsN):0;return `<div class="score"><b>${w} of ${rowsN}</b> <span>${s.n} is best</span><div class="bar"><i style="width:${pct}%"></i></div></div>`}).join("");
 const L=GL[g]||["",g];
 return `<div class="card fade"><div class="gtitle"><h2>${L[0]}</h2><span class="tag bl">${L[1]}</span><span class="tag">${sels.length} Bajaj · ${comps.length} competitors</span></div>
 <div class="scores" style="margin-bottom:14px">${sc}</div>${comps.length?"":`<p class="hint">No competitor data for this model in the uploaded files, so only its own specifications are shown.</p>`}
 <div class="legend"><span class="sw"></span>Best value for that parameter &nbsp;·&nbsp; ▲ higher is better &nbsp;·&nbsp; ▼ lower is better</div>
 <div class="wrap"><table><thead><tr><th class="pl">Parameter</th>${cols.map(c=>`<th class="${c.bj?"bj":""}"><small>${c.b} · ${c.fuel}</small>${c.n}</th>`).join("")}</tr></thead><tbody>${tb}</tbody>
 <tfoot><tr><td class="pl">Parameters where best</td>${cols.map(c=>`<td>${wins[c.id]}</td>`).join("")}</tr></tfoot></table></div></div>`}
function results(){$("cnt").textContent=sel.size?sel.size+" selected":"None selected yet. Tap any model.";
 const s=bj.filter(r=>sel.has(r.id));
 if(!s.length){$("res").innerHTML=`<div class="card empty"><b>No Bajaj model selected</b>Pick one or more models above and their competitors will appear here.</div>`;return}
 const gs=[...new Set(s.map(r=>r.g))];
 $("res").innerHTML=`<div class="opts"><label><input type="checkbox" id="hide" ${hide?"checked":""}> Hide parameters with no data</label></div>`+gs.map(g=>group(g,s.filter(r=>r.g==g))).join("");
 $("hide").onchange=e=>{hide=e.target.checked;results()}}
$("clr").onclick=()=>{sel.clear();picker();results()};
picker();results();
</script></body></html>""""""


# ============================================================ build + run
def build_page(records):
    data = json.dumps(records, ensure_ascii=False, separators=(",", ":"))
    return PAGE.replace("__DATA__", data.replace("</", "<\\/"))   # a stray '</script>' in a cell can't break the page


def run(extra_dir=None, out_path="Bajaj_Benchmarking.html", height=1000):
    """"""Find the workbooks, build the dashboard, save it and (in a notebook) display it. Silent unless logging is enabled.""""""
    try:
        import openpyxl  # noqa: F401
    except ImportError as e:
        raise ImportError("openpyxl is required: pip install openpyxl") from e

    paths = find_files([extra_dir] if extra_dir else [])
    for key, label in LABELS.items():
        log.info("%s %s%s", "found  " if paths[key] else "missing", label, f" -> {paths[key]}" if paths[key] else "")
    if not any(paths.values()):
        raise FileNotFoundError("No benchmarking workbook found. Pass the folder: run('/path/to/folder')")

    records = load_records(paths)
    page = build_page(records)
    try:
        with open(out_path, "w", encoding="utf-8") as fh:
            fh.write(page)
        log.info("%d vehicles (%d Bajaj) saved to %s", len(records), sum(r["bj"] for r in records), out_path)
    except OSError:
        log.warning("Could not save an HTML copy in this folder.")

    try:                                   # notebook / hosted view; harmless elsewhere
        from IPython.display import HTML, display
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:0;border-radius:12px"></iframe>'
                         % (html.escape(page, quote=True), height)))
    except ImportError:
        pass


run()"""


<>:24: SyntaxWarning: invalid escape sequence '\s'
<>:386: SyntaxWarning: invalid escape sequence '\d'
<>:399: SyntaxWarning: invalid escape sequence '\d'
<>:24: SyntaxWarning: invalid escape sequence '\s'
<>:386: SyntaxWarning: invalid escape sequence '\d'
<>:399: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_6700/3583430908.py:24: SyntaxWarning: invalid escape sequence '\s'
  s = re.sub(r'\s+', ' ', str(v).replace('\u200b', '').replace('\xa0', ' ')).strip()
/tmp/ipykernel_6700/3583430908.py:386: SyntaxWarning: invalid escape sequence '\d'
  _SIMPLE = re.compile(r'[\d.,]+\s*[A-Za-z/%°]*')
/tmp/ipykernel_6700/3583430908.py:399: SyntaxWarning: invalid escape sequence '\d'
  v = re.sub(r'(^|; )\d\.\s+', r'\1', re.sub(r'\s*[\r\n]+\s*', '; ', v.strip()))


'\nimport re, math, os, json, html, logging, warnings\n\nlog = logging.getLogger("bajaj_benchmarking")\n\n\n# ============================================================ cell access & parsing helpers\ndef sheet(wb, name):\n    for ws in wb.worksheets:\n        if ws.title.strip().lower() == name.strip().lower(): return ws\n    raise KeyError(f"Sheet \'{name}\' not found. Sheets: {[w.title for w in wb.worksheets]}")\ndef G(ws, coord):\n    c = ws[coord]\n    for m in ws.merged_cells.ranges:\n        if coord in m: return ws.cell(m.min_row, m.min_col).value\n    return c.value\ndef S(v):\n    if v is None: return None\n    s = re.sub(r\'\\s+\', \' \', str(v).replace(\'\u200b\', \'\').replace(\'\xa0\', \' \')).strip()\n    if not s or s.startswith(\'#\') or s.lower() in (\'-\', \'--\', \'---\', \'na\', \'n/a\', \'nan\', \'tbc\', \'to be confirmed\', \'to be checked\', \'not provided\'): return None\n    return s\ndef f(s):\n    s = S(s)\n    if s is None: return None\n    m = re.search(r